# Piece of my Heart

## Overview

This module implements the piece system for the hex-based strategy game, including:
- **Piece simulation** - Calculate future movement paths based on instruction rules
- **Visual overlays** - Render piece positions and planned movements on the map
- **Interactive controls** - Facing selector widget for piece orientation
- **Route handlers** - Web endpoints for piece detail views and maps

## Architecture

### Three-Layer Design

1. **Kingdom Directive** — Strategic intent (FEED, EXPLORE, EXPAND, DEFEND, ATTACK)
2. **Piece Orders** — Directive mapped onto each piece based on type and stats
3. **Moves** — Resolved outcomes after all kingdoms' orders collide


I am going to work through some thoughts

## Modified Gosper
so the rules on Piece is going to be reworked as a list of ints with the following
0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
4 -> settle or siege # reserved for figuring out how to conquer settlements
5 -> forward # move on place forward (depending on movement cost). If you can't move forward (ie you hit water) rotate left. if it is there is another piece in the way pause   
6 -> rotate left
7 -> rotate right


## Movement costs

we have a movement cost formula
```python
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost
```
Which we will need to figure out how to use for a turn. movement_range will tell how many moves we can do. Lets make turns free (including turns from hitting the ocean). the first 4 counts as a single movement cost.

## food and diet
we are going to add three fields 
1. which is the amount of food a piece has
2. its diet. how much does it eat per turn If it rounds out of food it will start to lose health. 
3. food capacity which is how much total food it can carry.

## deprecate
we don't need memory since that will be in rules.

## settle
We are going to keep these around even though I haven't figure them out
harvest_goal: Resources = None
settle_progress: int = 0
settle_threshold: int = 3
sight = 3

## The things we carry
Lets start thinking about the rules editor for a piece
1. we are going to want a list of rules that are assigned and be able to add and delete rows
2. we are going to want an edit point where we can insert new items
3. we are going to want a HexDragMap that we can add items. The path we make needs to be converted into instructions on how we get there.
4. we need an apply button and the number of rounds. This will be our main map showing where the piece is going to move each turn for the number of turns. I envision this as a series of arrows and circles.
a. arrows are for movement. straight line if single cost squiggle line if multiple cost. the color of the arrow is how aggressive it is.
b. circles are for the pause actions (0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
) I think red should be for defend.

eventually we are going to want a drop down of a list of common rule sets (patrol around a circle. go out and harvest and comeback, attack by first flanking left). 
we will certainly want to have a hex destination that returns a set of rules on how to get there (hello djisktra)

I need to be clearer on some definitons
1. `round` is for all player and all pieces. Each player gives the set of orders it wants its pieces to do. We need to build out how to carry them out, but I am tempted just to start with the kindergaren model of everybody does what it wants
2. `turn` is equivalent to a round, but just on a single piece. `movement_cost` is how many things it can do in a turn. Some (like rotation) are free. Others like mmoving up a hill might cost a couple of movements.

It can be the case that rule ends and there are spare items. So if the last item in the list is one of the non movement items. ie (like pause) then we just pause the remaining amount of time. if it is a movement (or rotate) then we circle back to the begining.

## Pulling Ranks
so things are prioritzed in chess order ie kings move first and eat first. then queens. This way if a king will move to a spot before a pawn would

I am going to have the flow of the hex be the amount you can harvest. Settlments are going to have build queues. I need to figure out the mechanics of adding settlements whether there are pre existing locations or it behavior on a item

**7. Food economy**

Ill need a `hex_food_yield(terrain, idx) -> float` keyed off climate and flow.



**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times but each time if the chance to move becomes available



Here's where we've landed:

**Budget model** — `move_strength` (rename to `speed`) is the per-turn movement budget, not per-instruction.

| Instruction | Cost | Notes |
|---|---|---|
| `FORWARD` (5) | 1 flat/downhill, 2 uphill | Can't move if budget < cost → turn ends |
| `ROT_L` (6) / `ROT_R` (7) | free | Including auto-rotations from hitting water |
| `PAUSE` (0) | 1 | Explicit wait |
| `DEFEND` (1) | 1 | Boosts combat response |
| `HARVEST` (2) | 1 | Gathers `hex_food_yield` into `food` (capped at `food_capacity`) |
| `GIVE` (3) | 1 | Transfer food to adjacent piece or settlement |
| `SETTLE` (4) | 1 | Reserved — siege/founding |

**FORWARD special cases:**
- **Water/edge** → auto-rotate left (free), up to 5×. Full circle → stuck, forced PAUSE (costs 1)
- **Occupied hex** → forced PAUSE (costs 1), cursor still advances

**End-of-rules:**
- Last instruction is non-movement (`PAUSE`/`DEFEND`/`HARVEST`/`GIVE`/`SETTLE`) → idle out remaining budget
- Last instruction is movement/rotation → wrap cursor to 0, keep going

**Turn resolution order:**
- KING → QUEEN → BISHOP → KNIGHT → ROOK → PAWN (chess rank priority)

**Food economy:**
- `food` — current supply
- `diet` — consumed per turn (deducted at turn end)
- `food_capacity` — max carried
- `food < 0` → lose health

**Example: Pawn (speed=3), rules `F F F F F F`, flat terrain:**
```
Turn 1: F(1) F(1) F(1) → budget 0
Turn 2: F(1) F(1) F(1) → budget 0
```

**Example: Pawn (speed=3), rules `F F F F F F`, uphill at step 2:**
```
Turn 1: F(1) F_uphill(2) → budget 0 (only 2 hexes moved)
Turn 2: F(1) F(1) F(1)  → budget 0
```

**Example: Pawn (speed=3), rules `F F HARVEST`:**
```
Turn 1: F(1) F(1) HARVEST(1) → end of rules, last=HARVEST → idle
Turn 2: same pattern (wraps because patrol=True)
```



In [ ]:
#| default_exp game/piece

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass
from datetime import datetime
from functools import cached_property
import heapq
import io
from enum import Enum

In [ ]:
#| export
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion

from HexMagic.overlay import OverlaySpec, FlowOverlay

In [ ]:
#| export
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point, Squad
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

## Piece View — Project Plan
That's a really clean three-layer design:

1. **Kingdom Directive** — strategic intent ("we're starving", "explore", "expand")
2. **Piece Orders** — the directive gets mapped onto each piece based on its type and base stats
3. **Moves** — resolved outcomes after all kingdoms' orders collide

So the directive acts like the queen bee's pheromone signal — it shifts *what* each piece does, but *how well* they do it still depends on their base stats. A "gather food" directive would assign harvest orders to pawns (who are great at it) but might also reassign knights to patrol the borders instead of scouting.

For the directive enum, something like:

- **FEED** — prioritize harvest
- **EXPLORE** — prioritize scouting / vision
- **EXPAND** — claim new territory
- **DEFEND** — protect capital and settlements
- **ATTACK** — target enemy pieces/settlements

These look right. So we would to combine them to make a patrol concept. we could have a gosper curve that a piece would through as it moves. if we see something and it wants to attack then it could

That's a brilliant use of the Gosper curve — it's space-filling, so a piece on patrol would efficiently cover an area without a lot of backtracking. The concept would be something like:

    Generate patrol path — a Gosper curve anchored to a settlement or border, sized to fit the piece's territory
    Each turn — the piece advances along the curve by move_strength hexes
    At each hex — check sight range for triggers (enemy pieces, unclaimed resources, etc.)
    Interrupt — if something is spotted, the piece breaks off patrol and generates a new order (attack, harvest, scout closer)

So a patrol is really a streaming order — it keeps going until something more interesting happens. The directive level would control where the patrol is anchored and what triggers a break:

    DEFEND → patrol around capital, interrupt on enemy sighting → attack
    EXPLORE → patrol outward from borders, interrupt on discovery → report back
    FEED → patrol farmable land, interrupt on good hex → harvest


Nice. So let's sketch the Order shape. Something like:
Copied!

class OrderType(Enum):
    PATROL = "patrol"
    ATTACK = "attack"
    MOVE   = "move"
    HARVEST = "harvest"
    SCOUT  = "scout"




In [ ]:
#| export
app, rt, hexGameServer  = appRoutes()

## Helpers

In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
#!cat ../../HexMagic/game/data.py

In [ ]:
#!cat ../../docs/ll*

### 1. Data Structures

In [ ]:
#| export
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int               # global step counter (across all turns)
    turn: int               # which turn (0-indexed)
    cost: float             # movement points consumed by this step
    budget_remaining: float # budget left after this step
    blocked: bool = False
    auto_rotations: int = 0 # free rotations from hitting water/edge

    @staticmethod
    def steps_by_turn(steps: list['PieceStep']) -> dict[int, list['PieceStep']]:
        """Group steps by turn number."""
        turns = {}
        for s in steps:
            turns.setdefault(s.turn, []).append(s)
        return turns


# Cost categories
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND, Instruction.HARVEST,
              Instruction.GIVE, Instruction.SETTLE}
_FREE = {Instruction.ROT_L, Instruction.ROT_R}

def _move_cost(elevations, from_idx, to_idx) -> float:
    """Simple movement cost: 1 flat/downhill, 2 uphill."""
    diff = elevations[to_idx] - elevations[from_idx]
    return 2.0 if diff > 0 else 1.0


In [ ]:
#| export
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    il = self.instructions
    if not il.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, il.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(il.rules):
                last = Instruction(il.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                if il.patrol:
                    cur = 0  # wrap around
                else:
                    break    # one-shot: stop entirely

            instr = Instruction(il.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac
                budget = max(0, budget - 1.0)
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget,
                                       blocked=True, auto_rotations=6))
                step_n += 1; cur += 1

    return steps


In [ ]:
#| export


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


### 2. Glyphs

In [ ]:
#| export
@dataclass
class DiagramGlyphs:
    """Shared SVG glyph renderer for piece-plan overlays."""
    color:        str   = "#333"
    stroke_width: float = 1.5
    opacity:      float = 0.85
    size:         float = 10.0


@patch
def circle(self: DiagramGlyphs, cx, cy, r, fill="none", stroke=None,
           stroke_width=None, opacity=None, **kw):
    stroke = stroke or self.color
    stroke_width = stroke_width or self.stroke_width
    opacity = opacity if opacity is not None else self.opacity
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')


@patch
def ring(self: DiagramGlyphs, cx, cy, r, stroke=None, stroke_width=2, opacity=0.9):
    return self.circle(cx, cy, r, fill="none",
                       stroke=stroke or self.color,
                       stroke_width=stroke_width, opacity=opacity)


@patch
def arc(self: DiagramGlyphs, cx, cy, r, start_deg, end_deg,
        stroke=None, stroke_width=2, opacity=0.8):
    import math
    stroke = stroke or self.color
    s, e = math.radians(start_deg), math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 '
            f'{x2:.1f},{y2:.1f}" fill="none" stroke="{stroke}" '
            f'stroke-width="{stroke_width}" opacity="{opacity}" '
            f'stroke-linecap="round"/>\n')


@patch
def shield(self: DiagramGlyphs, cx, cy, size=None, fill=None,
           stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')


@patch
def sheaf(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = ""
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def gift(self: DiagramGlyphs, cx, cy, size=None, fill=None,
         stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg


@patch
def house(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'opacity="{opacity}"/>\n')
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def pause(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=0.7):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    w, h = size * 0.2, size * 0.6
    svg = ""
    for dx in [-size*0.18, size*0.18]:
        svg += (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                f'width="{w:.1f}" height="{h:.1f}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
    return svg


@patch
def rot_arrow(self: DiagramGlyphs, cx, cy, r=None, clockwise=True,
              stroke=None, stroke_width=2, opacity=None):
    import math
    r = r or self.size * 0.5
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    if clockwise:
        arc_svg = self.arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc_svg = self.arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc_svg + ah


In [ ]:
#| export
@patch
def blocked_x(self: DiagramGlyphs, cx, cy, r=None) -> str:
    """Red circle with white X cross — blocked/impassable move marker."""
    r = r or self.size * 0.25
    svg = self.circle(cx, cy, r, fill="red", stroke="darkred",
                      stroke_width=1, opacity=0.55)
    d = r * 0.55
    for x1, y1, x2, y2 in [(cx-d, cy-d, cx+d, cy+d),
                             (cx+d, cy-d, cx-d, cy+d)]:
        svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
                f'x2="{x2:.1f}" y2="{y2:.1f}" '
                f'stroke="white" stroke-width="{max(1, r*0.35):.1f}" '
                f'stroke-linecap="round"/>\n')
    return svg


@patch
def start_marker(self: DiagramGlyphs, cx, cy, r=None,
                 stroke=None, stroke_width=None) -> str:
    """Hollow ring drawn around the piece's start position."""
    r = r or self.size * 0.8
    return self.ring(cx, cy, r,
                     stroke=stroke or self.color,
                     stroke_width=stroke_width or max(1.5, self.size * 0.07))


@patch
def step_label(self: DiagramGlyphs, cx, cy, n: int,
               offset=None, font_size=None, color=None) -> str:
    """Small step-number badge offset up-right from a hex centre.
    
    Args:
        n: step number (0-indexed internally; displayed as n+1)
        offset: pixel offset from centre (defaults to size * 0.35)
        font_size: override font size (defaults to max(12, size * 0.3))
        color: text color (defaults to self.comp if present, else self.color)
    """
    offset    = offset    or self.size * 0.35
    font_size = font_size or max(12, self.size * 0.3)
    color     = color     or getattr(self, 'comp', self.color)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{n+1}</text>\n')


@patch
def action_glyph(self: DiagramGlyphs, instr: Instruction,
                 cx, cy, fill=None, stroke=None) -> str:
    """Dispatch a non-movement instruction to its geometry glyph.
    
    Returns empty string for movement instructions (FORWARD) — those
    are handled separately as arrows.
    """
    fill   = fill   or self.color
    stroke = stroke or getattr(self, 'comp', self.color)
    size   = self.size

    dispatch = {
        Instruction.PAUSE:   lambda: self.pause(cx, cy, size, fill, stroke),
        Instruction.DEFEND:  lambda: self.shield(cx, cy, size, fill, stroke),
        Instruction.HARVEST: lambda: self.sheaf(cx, cy, size, fill, stroke),
        Instruction.GIVE:    lambda: self.gift(cx, cy, size, fill, stroke),
        Instruction.SETTLE:  lambda: self.house(cx, cy, size, fill, stroke),
        Instruction.ROT_L:   lambda: self.rot_arrow(cx, cy, clockwise=False,
                                                     stroke=stroke),
        Instruction.ROT_R:   lambda: self.rot_arrow(cx, cy, clockwise=True,
                                                     stroke=stroke),
    }
    fn = dispatch.get(instr)
    return fn() if fn else ""


### 3. Instruction extensions

In [ ]:
#| export
INSTR_LABELS = {
    Instruction.PAUSE:   ("Pause",      "⏸"),
    Instruction.DEFEND:  ("Defend",     "🛡"),
    Instruction.HARVEST: ("Harvest",    "🍆"),
    Instruction.GIVE:    ("Give",       "🎁"),
    Instruction.SETTLE:  ("Settle",     "🏰"),
    Instruction.FORWARD: ("Forward",    "→"),
    Instruction.ROT_L:   ("Turn Left",  "↺"),
    Instruction.ROT_R:   ("Turn Right", "↻"),
}


def _instr_glyph(instr: Instruction, size=26, color="#444"):
    """Inline SVG glyph for one instruction."""
    g = DiagramGlyphs(color=color, size=size * 0.55)
    cx, cy = size / 2, size / 2
    if instr == Instruction.FORWARD:
        body = (f'<line x1="{cx-size*0.28:.1f}" y1="{cy:.1f}" '
                f'x2="{cx+size*0.2:.1f}" y2="{cy:.1f}" '
                f'stroke="{color}" stroke-width="2.5" stroke-linecap="round"/>'
                f'<polygon points="{cx+size*0.28:.1f},{cy:.1f} '
                f'{cx+size*0.12:.1f},{cy-size*0.13:.1f} '
                f'{cx+size*0.12:.1f},{cy+size*0.13:.1f}" fill="{color}"/>')
    else:
        body = g.action_glyph(instr, cx, cy) or ""
    return NotStr(f'<svg width="{size}" height="{size}" viewBox="0 0 {size} {size}" '
                  f'style="vertical-align:middle">{body}</svg>')


In [ ]:
#| export
# ── Rendering ─────────────────────────────────────────────
@patch
def __ft__(self: InstructionList):
    """Full MonsterUI table with SVG glyphs and cursor marker."""
    if not self.rules:
        return P("No instructions", cls=TextPresets.muted_sm)

    header = ["#", "", "Action"]
    rows   = []
    for i, r in enumerate(self.rules):
        instr = Instruction(r)
        label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
        step = Strong("▶ ", cls="text-primary") if i == self.cursor else str(i + 1)
        rows.append([step, _instr_glyph(instr), label])

    tbl  = TableFromLists(header, rows, cls=(TableT.sm, TableT.striped, TableT.hover))
    mode = "🔁 Patrol (loops)" if self.patrol else "⏹ One-shot"
    return Card(tbl, footer=Small(mode, cls=TextPresets.muted_sm))

@patch
def compact(self: InstructionList, piece_id: str = "") -> FT:
    """Compact inline badge strip — lightweight alternative to the full table."""
    target_id = f"rules-{piece_id[:8]}" if piece_id else "rules-preview"

    if not self.rules:
        return P("No rules yet — drag a path above",
                    cls="text-sm opacity-50", id=target_id)

    _COLORS = {
        0: "#94a3b8", 1: "#ef4444", 2: "#22c55e",
        3: "#a855f7", 4: "#f59e0b", 5: "#3b82f6",
        6: "#64748b", 7: "#64748b",
    }
    badges = []
    for r in self.rules:
        instr  = Instruction(r)
        label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
        color  = _COLORS.get(r, "#888")
        badges.append(
            Span(label,
                    style=(f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                        f"color:{color}; margin:2px; display:inline-block;"))
        )

    return Div(
        P(f"{len(self.rules)} instructions", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        id=target_id,
    )

In [ ]:
#| export
@patch
def delete_at(self: InstructionList, idx: int):
    """Remove instruction at idx, clamping cursor."""
    if 0 <= idx < len(self.rules):
        self.rules.pop(idx)
        self.cursor = min(self.cursor, len(self.rules))

@patch
def insert_rules(self: InstructionList, new_rules: list[int]):
    """Insert rules at cursor position, advance cursor past them."""
    for i, r in enumerate(new_rules):
        self.rules.insert(self.cursor + i, r)
    self.cursor += len(new_rules)

@patch
def set_cursor(self: InstructionList, idx: int):
    self.cursor = max(0, min(idx, len(self.rules)))

@patch
def toggle_patrol(self: InstructionList):
    self.patrol = not self.patrol


In [ ]:
#| export
@patch
def est_turns(il: InstructionList, speed: float) -> int:
    """Lower-bound turn estimate: rotations are free, everything else ≥1 budget.
    Uphill FORWARDs cost 2 but terrain isn't available here, so this may
    undercount on hilly paths — conservative in the right direction.
    """
    free = {Instruction.ROT_L.value, Instruction.ROT_R.value}
    non_free = sum(1 for r in il.rules if r not in free)
    return max(1, math.ceil(non_free / max(float(speed), 1.0)))


In [ ]:
rules = [Instruction.FORWARD.value, Instruction.FORWARD.value,
         Instruction.ROT_L.value, Instruction.HARVEST.value, Instruction.SETTLE.value,
         Instruction.FORWARD.value, Instruction.ROT_R.value]

show(InstructionList(rules, cursor=2, patrol=True))


### 4. Piece extensions

In [ ]:
#| export
@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             suffix: str = "") -> str:
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes) or not self.flag:
        return ""

    center = grid.hexes[idx].center
    if scale is None:
        scale = (grid.radius * 0.75) / 22.5

    # Auto-select tier from scale
    size = 'board' if scale < 0.6 else ('list' if scale < 1.5 else 'large')

    return self.flag.draw_piece(
        self.piece_type, center, grid.builder,
        scale=scale, size=size,
        piece_id=f"piece_{self.id[:8]}{suffix}",
        opacity=opacity,
    )


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")


In [ ]:
#| export
def StatBar(label, current, maximum, color="auto"):
    """A labeled progress bar for bounded stats like health/food."""
    pct = min(100, max(0, (current / maximum * 100))) if maximum else 0
    if color == "auto":
        color = "success" if pct > 60 else "warning" if pct > 30 else "error"
    return Div(
        DivFullySpaced(
            P(label, cls=TextPresets.muted_sm),
            P(f"{int(current)}/{int(maximum)}", cls="text-xs font-mono"),
        ),
        Progress(value=str(int(pct)), max="100",
                 cls=f"uk-progress h-2 [&::-webkit-progress-value]:bg-{color}"),
        cls="space-y-0.5",
    )

@patch
def __ft__(self: Piece):
    """FastHTML card component for piece detail panel."""
    icon = _PIECE_ICONS.get(self.piece_type, "?")
    ptype = self.piece_type.name.title()
    short_id = self.id[:12] + "…"
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # Progress bars for depletable stats
    bars = [
        StatBar("Health", self.health, self.max_health),
        StatBar("Food", self.food, self.food_capacity),
    ]

    # Numeric stats
    num_stats = [
        ("⚔ Attack",  self.attack_strength),
        ("🏃 Speed",   self.move_strength),
        ("🌾 Harvest", self.harvest_strength),
        ("👁 Sight",   self.sight),
    ]
    stat_grid = Grid(
        *[Div(
            P(label, cls=TextPresets.muted_sm),
            P(str(val), cls="font-bold text-center"),
            cls="text-center",
        ) for label, val in num_stats],
        cols=4,
    )

    return Card(
        CardHeader(
            DivFullySpaced(
                DivLAligned(
                    P(icon, cls="text-3xl"),
                    Div(
                        H4(ptype, cls="font-bold"),
                        P(short_id, cls=TextPresets.muted_sm),
                    ),
                ),
                P(f"Facing {facing_arrow}", cls="text-lg"),
            ),
        ),
        CardBody(
            Div(*bars, cls="space-y-2"),
            Divider(),
            stat_grid,
            Divider(),
            InstructionList(self.rules, self.cursor, self.patrol),
        ),
        CardFooter(
            DivFullySpaced(
                Button("Edit Rules",
                       hx_get=f"/piece_rules/{self.id}",
                       hx_target="#right-panel",
                       cls=ButtonT.primary + " " + ButtonT.sm),
                Button("Show Plan",
                       hx_get=f"/piece_plan/{self.id}",
                       hx_target="#map",
                       cls=ButtonT.sm),
            ),
        ),
        id=f"piece-card-{self.id[:8]}",
    )


In [ ]:
#| export
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> set[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return set()

    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)

    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = HexPosition.origin().field_of_view( facing_dir, effective_sight)
        return {
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        }
    else:
        return set(grid.indices_in_range(self.location, effective_sight))


@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    squads: list[Squad],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Squad]:
    """Find pieces visible from piece's location, grouped by squad.

    Delegates visibility to hexes_in_sight, then filters squad members
    whose locations fall in the visible set.
    """
    visible = piece.hexes_in_sight(grid, elevations, elevation_mult, facing_only)
    if not visible:
        return []

    result = []
    for squad in squads:
        members = [
            p for p in squad.alive
            if p is not piece
            and p.location in visible
            and (not allied_only or p.owner_id == piece.owner_id)
        ]
        if members:
            result.append(Squad(id=squad.id, name=squad.name, pieces=members))
    return result


@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)
    
    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = field_of_view(HexPosition.origin(), facing_dir, effective_sight)
        return [
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        ]
    else:
        return grid.indices_in_range(self.location, effective_sight)

Can we rewrite hexes_in_sight and pieces in sight so that one of them takes advantage of the other?

## Build our testing envoirnment.

### 1. Piece Board

In [ ]:
#| export
@dataclass
class PieceBoard:
    """Lightweight playground for testing piece movement and overlays."""
    grid: HexGrid
    elevations: np.ndarray
    pieces: list  # list[Piece]
    countries: np.ndarray = None
    flags: dict = None  # countryId -> CountryFlag

    @classmethod
    def flat(cls, rings=5, radius=20, num_hexes_elev=200):
        """Open grassland — uniform elevation, no obstacles."""
        style = StyleCSS("pb_hex", fill="#e8e8e0", stroke="#ccc", stroke_width=0.5)
        grid = HexGrid.centered(rings=rings, radius=radius, style=style,
                                center=MapCord(radius*(2*rings+2), radius*(2*rings+2)))
        n = len(grid.hexes)
        elevations = np.full(n, num_hexes_elev, dtype=float)
        countries = np.zeros(n, dtype=float)
        # Mark invalid hexes as water
        for idx in grid.invalidRegion:
            elevations[idx] = 0
            countries[idx] = -1
        return cls(grid=grid, elevations=elevations, pieces=[],
                   countries=countries, flags={})

    @classmethod
    def hilly(cls, rings=5, radius=20, seed=42):
        """Rolling terrain with a river of water hexes through the middle."""
        board = cls.flat(rings=rings, radius=radius)
        rng = np.random.default_rng(seed)
        grid = board.grid
        n = len(grid.hexes)
        # Gentle elevation gradient + noise
        for i in range(n):
            if i in grid.invalidRegion: continue
            pos = grid.index_to_hexposition(i)
            board.elevations[i] = 150 + pos.q * 30 + rng.normal(0, 40)
            board.elevations[i] = max(1, board.elevations[i])
        # Carve a river: zero-elev stripe along r=0
        for i in range(n):
            if i in grid.invalidRegion: continue
            pos = grid.index_to_hexposition(i)
            if abs(pos.r) <= 0 and abs(pos.q) <= rings:
                board.elevations[i] = 0
                board.countries[i] = -1
        return board


In [ ]:
CountryFlag.commonNames??

In [ ]:
#| export
@patch
def add_piece(self: PieceBoard, hex_idx: int, piece_type=PieceType.PAWN,
              facing: int = 0, instructions: InstructionList = None,
              country_id: int = 1, birth_year: int = 1987) -> Piece:
    if instructions is None:
        instructions = InstructionList()
    if country_id not in self.flags:
        palette = CountryFlag.seaborn("husl", levels=max(country_id + 2, 4))
        self.flags[country_id] = palette[country_id]

    try:
        names = (CountryFlag.commonNames(year=birth_year, gender="M") +
                 CountryFlag.commonNames(year=birth_year, gender="F"))
        name = random.choice(names)
    except (ValueError, IndexError):
        name = "Marvin"

    piece = Piece(
        location=hex_idx, piece_type=piece_type,
        facing=facing,
        instructions=instructions,
        owner_id=country_id,
        name=name,
        birth_year=birth_year,
    )
    piece.flag = self.flags[country_id]
    self.pieces.append(piece)
    if self.countries is not None and self.countries[hex_idx] == 0:
        self.countries[hex_idx] = country_id
    return piece


In [ ]:
#| export
# ── Elevation palette for the playground ─────────────────────────────────────
_PB_WATER  = StyleCSS("pb_water",  fill="#6baed6", stroke="#4292c6", stroke_width=0.5)
_PB_LEVELS = [
    StyleCSS("pb_e0", fill="#c7e9c0", stroke="#bbb", stroke_width=0.5),  # low
    StyleCSS("pb_e1", fill="#a1d99b", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e2", fill="#74c476", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e3", fill="#d4b483", stroke="#bbb", stroke_width=0.5),  # hill
    StyleCSS("pb_e4", fill="#bc8a5f", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e5", fill="#9e6b4a", stroke="#999", stroke_width=0.5),  # peak
]

def _elev_style(elev: float, delta: float = 80.0) -> StyleCSS:
    """Map elevation to a terrain style."""
    if elev < 1: return _PB_WATER
    level = min(len(_PB_LEVELS) - 1, int(elev / delta))
    return _PB_LEVELS[level]


@patch
def _apply_terrain(self: PieceBoard):
    """Paint each hex with its elevation style, registering styles on the builder."""
    grid = self.grid
    for s in [_PB_WATER] + _PB_LEVELS:
        grid.builder.add_style(s)
    for i, hex_obj in enumerate(grid.hexes):
        if i in grid.invalidRegion: continue
        hex_obj.style = _elev_style(self.elevations[i])


@patch
def _pieces_overlay(self: PieceBoard) -> str:
    """SVG string: all pieces at their current locations."""
    svg = ""
    for p in self.pieces:
        svg += p.draw_svg(self.grid)
    return svg






In [ ]:
#| export
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#| export
def _turn_end_labels(steps: list[PieceStep]) -> list[tuple[int, int, int]]:
    """Merge consecutive turns that end on the same hex into ranges.
    
    Returns [(hex_idx, first_turn, last_turn), ...] — one entry per
    contiguous run of turns ending at the same hex.
    """
    # Find the hex where each turn ends
    turn_hex = {}  # turn -> hex_idx
    for s in steps:
        turn_hex[s.turn] = s.hex_idx

    if not turn_hex:
        return []

    sorted_turns = sorted(turn_hex)
    runs = []
    run_start = sorted_turns[0]
    prev_turn = run_start
    prev_hex  = turn_hex[run_start]

    for t in sorted_turns[1:]:
        if turn_hex[t] == prev_hex and t == prev_turn + 1:
            prev_turn = t  # extend the run
        else:
            runs.append((prev_hex, run_start, prev_turn))
            run_start = t
            prev_turn = t
            prev_hex  = turn_hex[t]
    runs.append((prev_hex, run_start, prev_turn))
    return runs


In [ ]:
#| export
def _range_label(g: DiagramGlyphs, cx, cy, t0, t1, color):
    """Emit '3' for a single turn or '2–4' for a range (1-indexed)."""
    text = str(t0 + 1) if t0 == t1 else f"{t0+1}–{t1+1}"
    offset    = g.size * 0.35
    font_size = max(12, g.size * 0.3)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{text}</text>\n')


In [ ]:
#| export
def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    if steps is None:
        if elevations is None: return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps: return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Find last step of each turn ──
    turn_ends = {}
    for i, s in enumerate(steps):
        turn_ends[s.turn] = i  # last one wins

    # ── Start marker ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for i, step in enumerate(steps):
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)
        elif step.blocked:
            svg += g.blocked_x(cx, cy)
        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        # Only label the last step of each turn
        #if i == turn_ends[step.turn]:
        #    svg += g.step_label(cx, cy, step.turn, color=comp)

        prev_idx = idx

    # ── Range labels (after the step loop) ──
    for hex_idx, t0, t1 in _turn_end_labels(steps):
        idx = _map_point(hex_idx, c2f)
        if 0 <= idx < N:
            cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y
            svg += _range_label(g, cx, cy, t0, t1, comp)


    return svg


In [ ]:
#| export
@patch
def _plan_overlay(self: PieceBoard, num_turns: int = 12) -> str:
    """SVG string: simulated movement plan for all pieces."""
    svg = ""
    for p in self.pieces:
        svg += piece_plan_overlay(
            p, self.grid, num_turns=num_turns,
            elevations=self.elevations, countries=self.countries,
        )
    return svg


In [ ]:
#| export
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_turns: int = 12, width: int = None, height: int = None) -> str:
    """Return a full SVG string of the board."""
    self._apply_terrain()
    grid = self.grid
    grid.update()

    grid.builder.adjust("pieces", self._pieces_overlay())
    grid.builder.adjust("plan", self._plan_overlay(num_turns) if show_plan else "")

    if width is None or height is None:
        xs = [h.center.x for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        ys = [h.center.y for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        pad = grid.radius * 1.5
        w = int(max(xs) + pad) if width  is None else width
        h = int(max(ys) + pad) if height is None else height
    else:
        w, h = width, height

    grid.builder.width  = w
    grid.builder.height = h
    return grid.builder.xml()


In [ ]:
#| export
@patch
def __ft__(self: PieceBoard):
    """Default view — terrain + pieces, no plan."""
    return Div(NotStr(self.render()))


def PieceBoardPlan(board: PieceBoard, num_turns: int = 12):
    """View with movement plan overlay."""
    return Div(NotStr(board.render(show_plan=True, num_turns=num_turns)))


In [ ]:
#| export
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg


In [ ]:
#| export
@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()

can you update our drawing methods to include the facing bar

In [ ]:
# ── Quick PieceBoard demo ─────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ]),
)

worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
        Instruction.ROT_L.value, Instruction.ROT_L.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.GIVE.value,
    ]),
)

guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
    ]),
)

bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
        Instruction.PAUSE.value,
        Instruction.SETTLE.value,
    ]),
)

PieceBoardPlan(board, num_turns=3)


Anything else you would change?A few things I'd tighten up, roughly in priority order:

**1. Ghost IDs aren't unique** — every ghost gets `id="piece_66ff270b_ghost"`, so you end up with duplicate IDs in the SVG. Some browsers get confused. Quick fix in `draw_ghost` / `draw_svg`:

```python
suffix=f"_ghost_{step_n}"  # or pass step index through
```

**2. Pattern defs are registered once per draw call** — look at the output: `piece_pat_66ff270b` appears as a `<pattern>` 15+ times. `_ensure_pattern` creates a new object each call; `add_definition` deduplicates by `id` but still runs every time. A one-liner cache would help:

```python
@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    if pat_name not in grid.builder.styles:  # or track in a set
        pat = self.flag.flagPattern(pat_name, scale=pat_scale)
        grid.builder.add_definition(pat)
    return f"url(#{pat_name})"
```

**3. `occupied` isn't threaded between pieces** — `_plan_overlay` calls `piece_plan_overlay` per piece but doesn't accumulate an `occupied` set across them. So two pieces can happily walk through each other in the preview. If you want collision-aware plans, build the set once and pass it through in chess-rank order.

**4. Budget can go negative** — in the "blocked" and "stuck" branches, `budget -= 1.0` fires without checking `budget >= 1.0`. If a piece has 0.5 budget left and gets blocked, `budget_remaining` becomes –0.5. A `budget = max(0, budget - 1.0)` guard would be cleaner.

**5. Food/diet isn't deducted yet** — the design says deduct `diet` at turn end and lose health when `food < 0`, but `simulate` doesn't touch `food` or `health`. Not a bug today since it's all read-only simulation, but worth a `# TODO` so you remember when you wire up actual round resolution.

The first two are the most immediately impactful — they affect every render right now.

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

## Editors

### 1. Piece Drag editor

In [ ]:
#| export
def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display compact badge strip.
    """
    base    = StyleCSS("pe-hex",   fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",   fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label", fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings=rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    grid.hexes[mid].style      = current
    grid.hexes[mid].label      = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        grid.builder.adjust("facing", grid.arrow(
            mid, facing_idx,
            style=StyleCSS("pe-arrow", stroke="#f59e0b", stroke_width=2),
        ))

    grid.update()

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=f"drag-editor-{piece.id[:8]}",
        ),
        piece.instructions.compact(piece.id),
        cls="space-y-2",
    )


In [ ]:
#| export
@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request: Request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return compact badge strip."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # Reconstruct the same grid used in piece_drag_editor
    rings, radius = 3, 35
    grid = HexGrid.centered(rings=rings, radius=radius,
                            style=StyleCSS("x", fill="white"))

    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist — append to existing rules
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.rules or "").split(",") if x]
            combined = existing + new_rules
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in combined)},
                pk="id",
            )
            new_rules = combined
    except Exception:
        pass  # demo mode — show without saving

    return InstructionList(new_rules).compact(piece_id)


### 2. Rule Edit Session

In [ ]:
#| export
@dataclass
class RuleEditSession:
    """In-memory edit buffer for a piece's rules."""
    piece_id: str
    instructions: InstructionList

_rule_sessions: dict[str, RuleEditSession] = {}

def _open_session(piece: Piece) -> RuleEditSession:
    sess = RuleEditSession(
        piece_id=piece.id,
        instructions=InstructionList(list(piece.rules), piece.cursor, piece.patrol),
    )
    _rule_sessions[piece.id] = sess
    return sess


In [ ]:
#| export
def _auth_piece(session, piece_id: str) -> tuple[RuleEditSession, int] | None:
    """Return (session, user_id) if the user owns this piece, else None.
    
    Checks:
      1. User is logged in (ensure_user)
      2. Piece belongs to the user's active world
      3. Piece's kingdom matches one the user controls
      4. An edit session exists (for mutation routes)
    """
    uid    = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return None

    # Does this piece belong to the active world?
    try:
        rec = globalStore.pieces[piece_id]
    except (KeyError, NotFoundError):
        return None

    r = rec if isinstance(rec, dict) else asdict(rec)
    if r.get("world_id") != active.world_id:
        return None

    # Does the piece's kingdom belong to this user's game?
    piece_kingdom = r.get("kingdom_id", 0)
    valid_kingdoms = {k.countryId for k in active.board.kingdoms}
    if piece_kingdom not in valid_kingdoms:
        return None

    return uid


def _auth_session(session, piece_id: str) -> RuleEditSession | None:
    """Like _auth_piece but also requires an active edit session."""
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return None
    return _rule_sessions.get(piece_id)


_DENIED = P("⛔ Not authorized", cls="text-sm text-red-500")
_EXPIRED = P("Session expired — reopen the editor", cls="text-sm text-red-500")


In [ ]:
#| export
# ── Shared grid factory ──────────────────────────────────────────────────────
def _editor_grid(rings=3, radius=35):
    return HexGrid.centered(
        rings=rings, radius=radius,
        style=StyleCSS("pe-hex", fill="#1e293b", stroke="#475569", stroke_width=1),
    )

def _editor_id(piece_id): return f"rule-editor-{piece_id[:8]}"


# ── Editor panel component ───────────────────────────────────────────────────
def _rules_editor(sess: RuleEditSession) -> FT:
    """Full server-rendered rules editor. Re-rendered on every mutation."""
    eid  = _editor_id(sess.piece_id)
    il   = sess.instructions
    rows = []

    for i, r in enumerate(il.rules):
        instr     = Instruction(r)
        label, _  = INSTR_LABELS.get(instr, (instr.name, "?"))
        is_cursor = (i == il.cursor)

        cursor_btn = Button(
            "▶" if is_cursor else "○",
            hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            cls=(ButtonT.primary if is_cursor else ButtonT.ghost) + " " + ButtonT.xs,
        )
        delete_btn = Button(
            UkIcon("x", height=12),
            hx_post=piece_rule_delete.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            hx_confirm="Delete this instruction?",
            cls=ButtonT.destructive + " " + ButtonT.xs,
        )
        rows.append([cursor_btn, _instr_glyph(instr), label, delete_btn])

    # "Append at end" cursor slot
    at_end = il.cursor >= len(il.rules)
    rows.append([
        Button("▶" if at_end else "○",
               hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=len(il.rules)),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               cls=(ButtonT.primary if at_end else ButtonT.ghost) + " " + ButtonT.xs),
        "", "— end —", "",
    ])

    tbl = TableFromLists(["", "", "Action", ""], rows,
                         cls=(TableT.sm, TableT.striped))

    patrol_toggle = LabelCheckboxX(
        "Patrol (loop)", checked=il.patrol,
        hx_post=piece_rule_patrol.to(piece_id=sess.piece_id),
        hx_target=f"#{eid}", hx_swap="outerHTML",
    )

    # Drag editor inserts at cursor
    grid = _editor_grid()
    grid.update()
    drag_section = Div(
        P(f"Drag to insert at position {il.cursor + 1}",
          cls="text-xs opacity-60 mb-1"),
        HexDragMap(grid, on_drag={
            "hx_post":   piece_path.to(piece_id=sess.piece_id, facing=0),
            "hx_target": f"#{eid}",
            "hx_swap":   "outerHTML",
        }),
    )

    action_row = DivFullySpaced(
        Button("Apply", cls=ButtonT.primary + " " + ButtonT.sm,
               hx_post=piece_rule_apply.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               hx_indicator=f"#{eid}"),
        Button("Cancel", cls=ButtonT.ghost + " " + ButtonT.sm,
               hx_post=piece_rule_cancel.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML"),
    )

    return Card(
        H5("Edit Rules", cls="font-bold"),
        tbl if il.rules else P("No instructions — drag a path below",
                                cls="opacity-50 text-sm"),
        patrol_toggle,
        Divider(),
        drag_section,
        Divider(),
        action_row,
        id=eid, cls="htmx-indicator",
    )




In [ ]:
#| export
@rt
def piece_rules(session, piece_id: str):
    """Open rules editor from current DB state."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    piece = globalStore.piece_from_id(piece_id)
    return _rules_editor(_open_session(piece))

@rt
def piece_rule_cursor(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.set_cursor(idx)
    return _rules_editor(sess)

@rt
def piece_rule_delete(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.delete_at(idx)
    return _rules_editor(sess)

@rt
def piece_rule_patrol(session, piece_id: str):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.toggle_patrol()
    return _rules_editor(sess)

@rt
async def piece_path(session, request, piece_id: str, facing: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    hex_ids = [int(x) for x in raw.split(",") if x.strip()]
    if hex_ids:
        grid = _editor_grid()
        new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)
        sess.instructions.insert_rules(new_rules)
    return _rules_editor(sess)

@rt
def piece_rule_apply(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    if not sess: return _EXPIRED

    il = sess.instructions
    globalStore.pieces.upsert({
        "id": piece_id,
        "rules": ",".join(str(r) for r in il.rules),
        "cursor": il.cursor,
        "patrol": 1 if il.patrol else 0,
    }, pk="id")

    eid = _editor_id(piece_id)
    editor_result = Div(
        P("✓ Rules saved", cls="text-sm text-green-600 font-bold mb-2"),
        il.compact(piece_id),
        id=eid,
    )
    map_update = Div(P("Map updated", cls="text-xs opacity-50"),
                     id="map", hx_swap_oob="innerHTML")
    return editor_result, map_update

@rt
def piece_rule_cancel(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    eid  = _editor_id(piece_id)
    if sess:
        return Div(sess.instructions.compact(piece_id), id=eid)
    return _EXPIRED


### 3. Direction Picker

In [ ]:
#| export
_DIR_ARROWS = ["↙", "←", "↖", "↗", "→", "↘"]   # keep

@rt
def set_facing(session, piece_id: str, facing: int):
    """Update piece facing — works with edit session if open."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED

    sess = _rule_sessions.get(piece_id)
    if sess:
        # If editor is open, update session (uncommitted)
        # facing lives on the Piece, not InstructionList — 
        # so we'd need to stash it. For now just re-render compass.
        pass

    facing = facing % 6
    target_id = f"facing-{piece_id[:8]}"

    # Reload piece, update, re-render compass
    piece = globalStore.piece_from_id(piece_id)
    piece.facing = facing
    globalStore.pieces.upsert({"id": piece_id, "facing": facing}, pk="id")

    return facing_grid(piece)


In [ ]:
#| export
def facing_grid(piece: Piece, radius: int = 30):
    """Compass rose: piece glyph in center, arrows pointing to each direction."""
    piece_id = piece.id
    base   = StyleCSS("fc-hex",    fill="#f8f8f8", stroke="#999", stroke_width=1.5)
    active = StyleCSS("fc-active", fill="#4CAF50", stroke="#333", stroke_width=2)
    center = StyleCSS("fc-center", fill="#e0e0e0", stroke="#666", stroke_width=1.5)

    grid = HexGrid.centered(1, radius=radius, style=base)
    for s in [active, center]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Direction hexes
    dir_map = {}
    for fi, d in enumerate(HexPosition.directions()):
        gi = grid.hexposition_to_index(d, mid)
        dir_map[gi] = fi
        grid.hexes[gi].style = active if fi == piece.facing else base
        grid.hexes[gi].label = _DIR_ARROWS[fi]
        grid.hexes[gi].labelStyle = "fc-label"

    grid.hexes[mid].style = center
    grid.builder.add_style(StyleCSS("fc-label", font_size="13px"))
    grid.update()

    # Overlay: piece in center + arrows from center to each neighbor
    overlay = piece.draw_svg(grid, hex_idx=mid, scale=(radius * 0.6) / 22.5)

    arrow_active = StyleCSS("fc-arr-on",  stroke="#4CAF50", stroke_width=2, fill="none")
    arrow_dim    = StyleCSS("fc-arr-off", stroke="#bbb",    stroke_width=1, fill="none")
    grid.builder.add_style(arrow_active)
    grid.builder.add_style(arrow_dim)

    for gi, fi in dir_map.items():
        style = arrow_active if fi == piece.facing else arrow_dim
        overlay += grid.arrow(mid, gi, style=style, factor=0.35)

    grid.builder.adjust("overlay", overlay)

    target_id = f"facing-{piece_id[:8]}"

    def on_click(g, i):
        if i in dir_map:
            return {
                "hx-post":   f"/set_facing/{piece_id}/{dir_map[i]}",
                "hx-target": f"#{target_id}",
                "hx-swap":   "outerHTML",
            }
        return {}

    return Div(
        H5("Facing", cls="font-bold text-sm mb-1"),
        HexTouchMap(grid, on_click=on_click),
        id=target_id,
        cls="inline-block",
    )


## Getting Data

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }

In [ ]:
def addMissing():
   

    # 1. Add the dummy user
    now = int(datetime.now().timestamp())
    globalStore.users.insert(dict(
        username="dummy", email="dummy@test.com", password="test",
        created=now, sessionID="", activeWorld=0, id=64801
    ))

    # 2. Create a world for them
    active = globalStore.create_game(64801, template_name="bayArea")

    # 3. Recruit a Knight into the first settlement of the first kingdom
    board = active.board
    kingdom = board.kingdoms[0]
    settlement = kingdom.settlements[0]
    settlement.db = globalStore  # wire up DB so save works

    knight = settlement.recruit(PieceType.KNIGHT, flag=kingdom.flag)
    knight.save(db=globalStore, world_id=active.world_id, kingdom_id=kingdom.countryId)

    print(f"User: 64801")
    print(f"World: {active.world_id}")
    print(f"Kingdom: {kingdom.countryName} (id={kingdom.countryId})")
    print(f"Settlement: {settlement.name} @ hex {settlement.location}")
    print(f"Knight: {knight.id[:12]}… facing={knight.facing} atk={knight.attack_strength} mov={knight.move_strength}")


In [ ]:
#addMissing()

In [ ]:
#| export
def showPieces():
    users_df = pd.DataFrame(globalStore.pieces())
    print("pieces:")
    print(users_df)

In [ ]:
showPieces()

In [ ]:
#| export
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret


In [ ]:
showActiveSettlements(dummySession)

In [ ]:
#| export
def listSettlements():
    users_df = pd.DataFrame(globalStore.settlements())
    print("pieces:")
    print(users_df)

In [ ]:
listSettlements()

In [ ]:
dummySettlment = "7ba2cc4b-97fd-4682-8d34-e4b9ea06cd72"

In [ ]:
#| export
@rt("/piece_map/{id}")
def piece_map(session, id: str, rings: int = None):
    """Map partial — renders zoomed terrain with active overlays."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")

    if rings is not None:
        rings = max(3, min(12, rings))
        session['settlement_rings'] = rings
    else:
        rings = session.get('settlement_rings', 5)

    overlays = get_overlays(session)

    try:
        place = globalStore.piece_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        piece_attrs = {
            'hx-get': f'/piece_detail/{id}',
            'hx-target': '#right-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        zoomed.apply_overlays(result, board, c2f, overlays, piece_attrs)

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


In [ ]:
#| export
def _wire_flag(piece: Piece, board: GameBoard):
    """Attach the correct kingdom flag to a loaded piece."""
    for k in board.kingdoms:
        if k.countryId == piece.owner_id:
            piece.flag = k.flag
            return


def _nearby_pieces(board: GameBoard, target: Piece, region: HexRegion) -> list[Piece]:
    """Other pieces whose current location falls within the visible region."""
    nearby = []
    for kingdom in board.kingdoms:
        for settlement in kingdom.settlements:
            for citizen in settlement.citizens:
                if citizen.id == target.id:
                    continue
                if citizen.location is not None and citizen.location in region.hexes:
                    if not citizen.flag:
                        citizen.flag = kingdom.flag
                    nearby.append(citizen)
    return nearby


def _render_piece_map(session, piece_id: str, num_turns: int = 6):
    """Core renderer: zoomed map centered on a piece with its movement plan.

    No occupied set → overlapping paths are intentionally visible.
    Other pieces in the region shown as static markers at 60% opacity.
    """
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return Div(_DENIED, id="map")

    active = globalStore.active_board(uid)
    if not active:
        return Div(P("No active game"), id="map")

    board     = active.board
    terrain   = board.terrain
    grid      = terrain.hexGrid
    countries = terrain.fields.get("country")

    piece = globalStore.piece_from_id(piece_id)
    _wire_flag(piece, board)

    if piece.location is None or piece.location < 0:
        return Div(P("Piece has no location"), id="map")

    # Simulate on coarse grid — collision-free so convergent paths show
    steps, region = piece.plan_region(
        grid, terrain.elevations,
        num_turns=num_turns,
        countries=countries,
        # occupied deliberately omitted
    )

    if not region.hexes:
        return Div(P("Empty region"), id="map")

    # Zoom into the padded bounding region
    result   = active.cover.zoom_region_fast(region, compute_weather=True)
    zterrain = result.terrain
    zgrid    = zterrain.hexGrid
    zgrid.adjustRadius(grid.radius)
    zgrid.builder.layers = []
    zterrain.colorMap()
    zgrid.update()
    c2f = result.c2f
    N   = len(zgrid.hexes)

    # 1. Country borders
    borders_svg = board.countries_overlay(zterrain, c2f)

    # 2. Other pieces in the area — static position markers
    others_svg = ""
    for other in _nearby_pieces(board, piece, region):
        idx = _map_point(other.location, c2f)
        if 0 <= idx < N:
            others_svg += other.draw_svg(zgrid, hex_idx=idx, opacity=0.6)

    # 3. Target piece's movement plan
    plan_svg = piece_plan_overlay(piece, zgrid, steps=steps, c2f=c2f)

    # Layer order: borders underneath, then other pieces, then plan on top
    zgrid.builder.adjust("borders", borders_svg)
    zgrid.builder.adjust("others",  others_svg)
    zgrid.builder.adjust("plan",    plan_svg)

    return Div(NotStr(zgrid.builder.xml()), id="map")


# ── Routes ────────────────────────────────────────────────────────────────────

@rt("/piece_map/{piece_id}")
def piece_map(session, piece_id: str, num_turns: int = 6):
    """Zoomed map centered on a piece showing its planned movement."""
    return _render_piece_map(session, piece_id, num_turns)


@rt("/piece_plan/{piece_id}")
def piece_plan(session, piece_id: str, num_turns: int = 6):
    """Alias — wired to the piece card's 'Show Plan' button."""
    return _render_piece_map(session, piece_id, num_turns)


@rt("/piece_detail/{piece_id}")
def piece_detail(session, piece_id: str):
    """Full piece stats card for the right panel."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED

    active = globalStore.active_board(uid)
    if not active: return P("No active game")

    piece = globalStore.piece_from_id(piece_id)
    _wire_flag(piece, active.board)
    return piece  # Piece.__ft__ renders the card with stats + rules + buttons


### FastHTML

In [ ]:
#| export
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")  # fallback: unicode glyph from PieceType.icon

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip ──
    rules_el = self.instructions.compact(self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )


In [ ]:
active = globalStore.active_board(64801)
piece = globalStore.piece_from_id("bbe865d0-1751-4076-a125-1d5e1f804ab5")
_wire_flag(piece, active.board)

# Give it some rules so the card isn't empty
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value, Instruction.FORWARD.value,
    Instruction.HARVEST.value,
]
piece.cursor = 0
piece.food = 3.5  # show the food bar partially filled

show(piece)


In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.ROOK, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()

In [ ]:
#| export
@patch
def badge(piece: Piece, size: int = 32) -> FT:
    """Colored circle with unicode chess glyph — readable at any size."""
    icon = piece.piece_type.icon
    if not piece.flag:
        return P(icon, cls="text-lg")
    bg, fg = piece.flag.primary, piece.flag.comp
    return Div(
        P(icon, style=f"color:{fg}; font-size:{size*0.55}px; line-height:1;"),
        style=(f"width:{size}px; height:{size}px; border-radius:50%; "
               f"background:{bg}; display:flex; align-items:center; "
               f"justify-content:center; border:2px solid {fg};"),
    )


In [ ]:
#| export
def PieceList(pieces: list[Piece], 
              hx_get_prefix: str = "/piece_detail",
              hx_target: str = "#right-panel",
              hx_swap: str = "innerHTML",
              icon_size: int = 32,
              selected_id: str = "",
              distances: dict[str, int] = None,
              **kwargs):
    """Compact clickable piece roster for sidebar panels.
    
    Each row: [flag-colored badge] Name  b.Year  ⬡dist  ♥Health
    Clicking a row loads that piece's detail card.
    """
    if not pieces:
        return P("No pieces", cls="text-sm opacity-50")

    distances = distances or {}
    rows = []
    for p in pieces:
        icon_el = p.badge(icon_size)

        # Health as colored text
        hp_pct = (p.health / p.max_health * 100) if p.max_health else 0
        hp_color = ("text-green-500" if hp_pct > 60
                    else "text-yellow-500" if hp_pct > 30
                    else "text-red-500")

        name = p.name or p.piece_type.name.title()
        year = f"b.{p.birth_year}" if p.birth_year else ""

        is_selected = p.id == selected_id
        sel_cls = ("bg-primary/10 border-primary" if is_selected
                   else "border-transparent hover:bg-base-300")

        # Distance badge (if available)
        dist_el = None
        if p.id in distances:
            d = distances[p.id]
            dist_el = P(f"⬡ distance {d}", cls="text-[10px] font-mono opacity-60 flex-shrink-0")

        row = Div(
            Div(icon_el, cls="flex-shrink-0"),
            Div(
                P(name, cls="text-sm font-bold truncate"),
                P(year, cls="text-[10px] opacity-50") if year else None,
                cls="flex-1 min-w-0",
            ),
            dist_el,
            P(f"♥{p.health}", cls=f"text-xs font-mono {hp_color} flex-shrink-0"),

            hx_get=f"{hx_get_prefix}/{p.id}",
            hx_target=hx_target,
            hx_swap=hx_swap,
            cls=f"flex items-center gap-2 px-2 py-1 rounded cursor-pointer border {sel_cls}",
        )
        rows.append(row)

    return Div(*rows, cls="space-y-1", **kwargs)


In [ ]:
#| export
def pieces_center(pieces: list[Piece], grid: HexGrid) -> tuple[int, dict[str, int]]:
    """Find the center hex of a group of pieces and each piece's distance to it.
    
    Returns (center_hex_idx, {piece_id: hex_distance}).
    Center is the cube-rounded average of all piece positions.
    """
    locs = [(p, grid.index_to_hexposition(p.location))
            for p in pieces
            if p.location is not None and p.location >= 0]
    if not locs:
        return -1, {}

    # Average cube coordinates
    n = len(locs)
    avg_q = sum(pos.q for _, pos in locs) / n
    avg_r = sum(pos.r for _, pos in locs) / n
    avg_s = sum(pos.s for _, pos in locs) / n

    # Cube-round: snap to nearest valid cube coord (q+r+s == 0)
    rq, rr, rs = round(avg_q), round(avg_r), round(avg_s)
    dq, dr, ds = abs(rq - avg_q), abs(rr - avg_r), abs(rs - avg_s)
    if   dq > dr and dq > ds: rq = -rr - rs
    elif dr > ds:              rr = -rq - rs
    else:                      rs = -rq - rr

    def _hex_dist(pos):
        return max(abs(pos.q - rq), abs(pos.r - rr), abs(pos.s - rs))

    # Closest valid grid index to the rounded center
    best_idx, best_d = -1, float('inf')
    for i in range(len(grid.hexes)):
        if i in grid.invalidRegion: continue
        d = _hex_dist(grid.index_to_hexposition(i))
        if d < best_d:
            best_d, best_idx = d, i

    # Per-piece distances
    distances = {p.id: _hex_dist(pos) for p, pos in locs}

    return best_idx, distances


In [ ]:
#| export
def GroupedPieceList(pieces: list[Piece],
                     country_names: dict[int, str] = None,
                     country_flags: dict[int, 'CountryFlag'] = None,
                     grid: HexGrid = None,
                     hx_get_prefix: str = "/piece_detail",
                     hx_target: str = "#right-panel",
                     hx_swap: str = "innerHTML",
                     icon_size: int = 32,
                     selected_id: str = "",
                     **kwargs):
    """Piece roster grouped by country. If grid is provided, shows hex distance from group center."""
    if not pieces:
        return P("No pieces", cls="text-sm opacity-50")

    country_names = country_names or {}
    country_flags = country_flags or {}

    # Compute distances from center if grid available
    distances = {}
    if grid is not None:
        _center_idx, distances = pieces_center(pieces, grid)

    # Group by owner_id
    groups = {}
    for p in pieces:
        groups.setdefault(p.owner_id, []).append(p)
    for cid in groups:
        groups[cid].sort(key=lambda p: (p.name or p.piece_type.name).lower())

    sections = []
    for cid in sorted(groups):
        name = country_names.get(cid, f"Country {cid}")
        flag = country_flags.get(cid)

        dot_color = flag.primary if flag else "#888"
        header = DivLAligned(
            Span(style=f"width:10px; height:10px; border-radius:50%; "
                       f"background:{dot_color}; display:inline-block; flex-shrink:0;"),
            P(name, cls="text-xs font-bold uppercase tracking-wide opacity-70"),
            cls="gap-2 pt-2 pb-1 px-1",
        )

        roster = PieceList(
            groups[cid],
            hx_get_prefix=hx_get_prefix,
            hx_target=hx_target,
            hx_swap=hx_swap,
            icon_size=icon_size,
            selected_id=selected_id,
            distances=distances,
        )
        sections.append(Div(header, roster))

    return Div(*sections, cls="space-y-2", **kwargs)


So we need to modify PieceList so it shows distance to the center of the group.

In [ ]:
??HexRegion

In [ ]:
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

# Country 1 — green-ish flag
scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ]),
)
worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
    ]),
)
queen = board.add_piece(
    mid + 1, piece_type=PieceType.QUEEN, facing=2, country_id=1,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.FORWARD.value,
    ]),
)

# Country 2 — red-ish flag
guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
    ]),
)
bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    instructions=InstructionList(rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value, Instruction.SETTLE.value,
    ]),
)

# Flavor
scout.food = 7.0
worker.food = 2.0; worker.health = 85
queen.food = 8.0
guard.food = 9.0; guard.health = 40
bishop.food = 5.5

# Country metadata
country_names = {1: "Thornwall Keep", 2: "Red Basin"}

show(Div(
    # Left panel — grouped piece roster
    Div(
        H5("Units", cls="font-bold mb-2"),
        GroupedPieceList(
            board.pieces,
            country_names=country_names,
            country_flags=board.flags,
            hx_get_prefix="/piece_detail",
            hx_target="#demo-detail",
            selected_id=scout.id,
        ),
        Divider(),
        Div(id="demo-detail"),
        cls="w-64 bg-base-200 p-3 flex-shrink-0 overflow-y-auto",
    ),

    # Main area — board with plan overlay
    Div(
        NotStr(board.render(show_plan=True, num_turns=3)),
        cls="flex-1 overflow-auto",
    ),

    cls="flex", style="height:550px;",
))


Lets build a function that finds using HexRegion the center of a list of pieces. and then have GroupedPieceList show the distance from the center

### On Terrain

In [ ]:
showDemo = True

In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the existing piece (the knight)
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Assign a patrol route: forward-forward-turn, with a harvest stop
piece.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
], cursor=0, patrol=True)


# Simulate on the coarse grid
steps, region = piece.plan_region(
    grid, terrain.elevations,
    num_turns=6, padding=4,
    countries=terrain.fields.get("country")
)

print(f"Piece: {piece.piece_type.name} at hex {piece.location}, facing={piece.facing}")
print(f"{len(steps)} steps, {len(region)} hex region\n")
for s in steps:
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. {s.instruction.name:8s} → hex {s.hex_idx}  f={s.facing}{tag}")

# Zoom into the plan region
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw the plan overlay on the zoomed grid
overlay = piece_plan_overlay(piece, zoomed.hexGrid,
                             steps=steps,
                             c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)
#webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


### On Text

In [ ]:
# Re-simulate with the fix applied
steps, region = piece.plan_region(grid, terrain.elevations,
                                  num_turns=5, padding=4,
                                  countries=terrain.fields.get("country"))

print(f"{len(steps)} steps, {len(region)} hex region")

# Zoom
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw plan overlay
overlay = piece_plan_overlay(piece, zoomed.hexGrid,
                             steps=steps, c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)
if showDemo:
    webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))
#zoomed.hexGrid.builder.show()


## Paths

In [ ]:
#| export
@dataclass
class TroopPath:
    """A resolved hex route with cached metadata."""
    hexes: list[int]
    cost: float = 0.0          # total movement cost (sum of _move_cost steps)
    
    def __len__(self):   return len(self.hexes)
    def __iter__(self):  return iter(self.hexes)
    def __bool__(self):  return len(self.hexes) > 1   # empty or single-hex = falsy
    def __getitem__(self, i): return self.hexes[i]
    
    @cached_property
    def as_set(self) -> set[int]:
        return set(self.hexes)
    
    @property
    def start(self) -> int: return self.hexes[0]
    
    @property
    def end(self) -> int:   return self.hexes[-1]
    
    def to_rules(self, grid: HexGrid, start_facing: int = 0) -> list[int]:
        return InstructionList.path_to_rules(self.hexes, grid, start_facing=start_facing)


In [ ]:
#| export
@patch
def pathfind(self: Piece, target: int, grid: HexGrid,
             elevations: np.ndarray,
             countries: np.ndarray = None, source:int=None) -> TroopPath:
    """Dijkstra from piece.location to target. Returns TroopPath (falsy if unreachable)."""
    if source is None:
        start = self.location
    else:
        start = source
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current == target:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)





@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> TroopPath:
    """Nearest reachable tile with food >= min_tier. Returns TroopPath (falsy if none)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)



In [ ]:
#| export
def TroopPathsOverlay(paths: list[TroopPath],
                      c2f: dict[int, int] = None,
                      colors: list[str] = None,
                      default_color: str = "#fff5e1",
                      stroke_width: float = 2.0,
                      opacity: float = 0.7,
                      dash: str = "6,3",
                      show_cost: bool = True,
                      font_size: int = 11,
                      priority: int = 50,
                      **kw) -> OverlaySpec:
    """Overlay showing TroopPath routes as dashed arrows on the hex grid.
    
    Args:
        paths: List of TroopPath objects to render
        c2f: Optional coarse-to-fine index mapping for zoomed views
        colors: Per-path colors (falls back to default_color)
        default_color: Color when no per-path color given
        stroke_width, opacity, dash: Styling for the route lines
        show_cost: If True, label each path endpoint with total cost
        font_size: Font size for cost labels
        priority: Overlay render priority
    """
    def _map(idx: int) -> int | None:
        """Map coarse index to fine index (identity if no c2f)."""
        if c2f is None: return idx
        return c2f.get(idx)
    
    def render(ctx: OverlayContext) -> str:
        grid = ctx.terrain.hexGrid
        parts = []
        
        for i, tp in enumerate(paths):
            if not tp: continue
            
            color = (colors[i] if colors and i < len(colors)
                     else default_color)
            
            # --- style for this path ---
            style = StyleCSS(f"troop_path_{i}",
                             stroke=color,
                             stroke_width=stroke_width,
                             fill="none",
                             opacity=opacity,
                             stroke_dasharray=dash)
            grid.builder.add_style(style)
            
            # --- draw arrow segments between consecutive hexes ---
            mapped = [_map(h) for h in tp.hexes]
            for j in range(len(mapped) - 1):
                src, dst = mapped[j], mapped[j + 1]
                if src is None or dst is None: continue
                parts.append(grid.arrow(src, dst, style=style,
                                        fromMiddle=True, factor=0.1))
            
            # --- start marker: filled dot ---
            start_fine = mapped[0]
            if start_fine is not None:
                sc = grid.hexes[start_fine].center
                parts.append(
                    f'<circle cx="{sc.x:.1f}" cy="{sc.y:.1f}" r="4" '
                    f'fill="{color}" opacity="{opacity}" />')
            
            # --- end marker: ring + optional cost label ---
            end_fine = mapped[-1]
            if end_fine is not None:
                ec = grid.hexes[end_fine].center
                parts.append(
                    f'<circle cx="{ec.x:.1f}" cy="{ec.y:.1f}" r="6" '
                    f'fill="none" stroke="{color}" stroke-width="2" '
                    f'opacity="{opacity}" />')
                
                if show_cost and tp.cost > 0:
                    parts.append(
                        f'<text x="{ec.x + 8:.1f}" y="{ec.y - 4:.1f}" '
                        f'text-anchor="start" font-size="{font_size}" '
                        f'fill="{color}" font-family="sans-serif" '
                        f'font-weight="bold">{tp.cost:.0f}</text>')
        
        return "\n".join(parts)
    
    return OverlaySpec("troop_paths", render, priority=priority)


In [ ]:
#| export
class SurroundMode(Enum):
    FEED  = "feed"   # face inward — supply chain
    GUARD = "guard"  # face outward — defensive perimeter


@patch
def effective_sight(self: Piece, elevations: np.ndarray,
                    elevation_mult: float = 0.005) -> float:
    """Sight range including elevation bonus."""
    if self.location is None or self.location < 0:
        return float(self.sight)
    elev = max(0, elevations[self.location])
    return self.sight + elev * elevation_mult


def _facing_toward(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) from from_idx toward to_idx."""
    rel = grid.index_to_hexposition(to_idx, from_idx)
    # Find closest cardinal direction
    dirs = HexPosition.directions()
    best_dir = 0
    best_dot = -999
    for i, d in enumerate(dirs):
        # Dot product in cube coords
        dot = rel.q * d.q + rel.r * d.r + rel.s * d.s
        if dot > best_dot:
            best_dot = dot
            best_dir = i
    return best_dir


def _facing_away(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) pointing away from to_idx."""
    return (_facing_toward(grid, from_idx, to_idx) + 3) % 6



In [ ]:
#| export

@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Thin wrapper: path to target → InstructionList."""
    path = self.pathfind(target, grid, elevations, countries)
    if not path:
        return InstructionList()
    return InstructionList(path.to_rules(grid, self.facing), cursor=0, patrol=False)

In [ ]:
#| export
@patch
def mapPath(self: TroopPath, grid: HexGrid,
                style: StyleCSS = None) -> MapPath:
    """Convert hex index route to a pixel-space MapPath for rendering."""
    if not self.hexes:
        return MapPath([], style or StyleCSS("troop", fill="none", stroke="#333", stroke_width=2))
    style = style or StyleCSS("troop", fill="none", stroke="#333", stroke_width=2)
    coords = [grid.hexes[idx].center for idx in self.hexes if 0 <= idx < len(grid.hexes)]
    return MapPath(coords, style)


In [ ]:
!cat ../../HexMagic/plot/*.py

I would love a @patch function on troopPath that takes a grid and returns a MapPath

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def to_map_path(self: TroopPath, grid: HexGrid,
                style: StyleCSS = None) -> MapPath:
    """Convert hex index route to a pixel-space MapPath for rendering."""
    if not self.hexes:
        return MapPath([], style or StyleCSS("troop", fill="none", stroke="#333", stroke_width=2))
    style = style or StyleCSS("troop", fill="none", stroke="#333", stroke_width=2)
    coords = [grid.hexes[idx].center for idx in self.hexes if 0 <= idx < len(grid.hexes)]
    return MapPath(coords, style)
```

That gives you the straight-line version. You can chain `.make_windy()` or `.smooth()` on the result since those are already patched onto `MapPath`:

```python
path = troop_path.to_map_path(grid, my_style).make_windy(iterations=2, offset_factor=0.1)
```

## Manuevers

```markdown
# Maneuvers & Formations — API Reference

## Core Primitives

### TroopPath

A resolved hex route with cached metadata. Returned by all pathfinding functions.

```python
path = piece.pathfind(target_hex, grid, elevations, countries=None)

path.hexes       # list[int] — ordered hex indices
path.cost        # float — total movement cost
path.start       # first hex
path.end         # last hex
path.as_set      # set[int] — cached for fast membership checks
path.to_rules()  # list[int] — convert to Instruction values

bool(path)       # False if empty or single-hex (unreachable)
len(path)        # number of hexes in route
```

**Movement costs:** flat/downhill = 1.0, uphill = 2.0. Water and invalid hexes are impassable.

### Pathfinding Variants

```python
# Basic A→B shortest path (Dijkstra)
path = piece.pathfind(target_hex, grid, elevations, countries)

# Nearest hex with food_tier >= threshold
path = piece.pathfind_to_food(grid, elevations, food_tiers, min_tier=3, countries)

# Path AWAY from a threat — prefers high ground in the opposite hemisphere
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)

# Convenience: pathfind + convert to InstructionList in one call
il = piece.pathfind_to_rules(target, grid, elevations, countries)
```

### InstructionList

The rule program for a piece. Each rule is an `Instruction` enum value.

```python
il = InstructionList(rules=[...], cursor=0, patrol=False)

il.insert_rules(new_rules)   # insert at cursor, advance cursor
il.delete_at(idx)             # remove one rule, clamp cursor
il.set_cursor(idx)            # move cursor
il.toggle_patrol()            # flip loop mode
il.est_turns(speed)           # lower-bound turn count for sync
il.compact(piece_id)          # FastHTML badge strip
```

**Patrol mode:** when `patrol=True`, cursor wraps to 0 at end of rules.
When `patrol=False`, piece stops after the last instruction.

### Instruction Budget Rules

Per turn, each piece gets `move_strength` budget points:

| Instruction | Cost | Notes |
|-------------|------|-------|
| FORWARD | 1 (flat/downhill) or 2 (uphill) | Into water/edge: auto-rotate left free, up to 5×. Into occupied hex: forced PAUSE cost 1. |
| ROT_L, ROT_R | 0 (free) | |
| PAUSE | 1 | |
| DEFEND | 1 | |
| HARVEST | 1 | |
| GIVE | 1 | |
| SETTLE | 1 | |

When budget runs out mid-instruction, the turn ends. If the last instruction
was non-movement (PAUSE/DEFEND/HARVEST/GIVE/SETTLE), remaining budget is
idled out as implicit PAUSEs.

---

## Formations

### FormationType

```python
class FormationType(Enum):
    LINE   = "line"    # row abreast, one step behind leader
    WEDGE  = "wedge"   # V-shape, leader at point, wings fan back
    BOX    = "box"     # ring-1 around leader (overflow to ring-2 if >6)
    COLUMN = "column"  # single file directly behind leader
```

**Choosing a formation:**
- **COLUMN** — narrow passages, rivers, mountain paths
- **LINE** — broad front for harvesting or sight coverage
- **WEDGE** — aggressive advance, concentrates force at point of contact
- **BOX** — protect a VIP (queen, settler), all-around defense

### form_up

Assemble followers around the leader's **current position**.

```python
orders = squad.form_up(
    FormationType.WEDGE, grid, elevations,
    leader=None,          # auto-picks highest rank if omitted
    countries=None,
    food_tiers=None,      # enables harvest padding during sync
    sync=True,            # pad so all arrive same turn
)
# orders: {piece_id: InstructionList} — leader NOT included
```

**Important:** `form_up` does not move the leader. It positions followers
relative to where the leader already is. Compose with `march_to` or
`pathfind_to_rules` for the leader's own movement.

### march_to

Move entire squad to a destination, arriving in formation.

```python
orders = squad.march_to(
    destination_hex,
    FormationType.LINE, grid, elevations,
    leader=None,          # auto-picks highest rank
    countries=None,
    food_tiers=None,
    sync=True,
)
# orders: {piece_id: InstructionList} — leader IS included
```

Formation offsets are computed relative to **destination + leader's arrival
facing**, so the formation assembles correctly on arrival. Uses Hungarian
matching to optimally assign followers to slots.

---

## Tactical Maneuvers

### surround

Position squad members around a target hex in a ring.

```python
orders = squad.surround(
    target_hex, grid, elevations,
    mode=SurroundMode.GUARD,  # GUARD = face outward, FEED = face inward
    max_ring=3,
    countries=None,
    elevation_mult=0.005,     # sight bonus per elevation unit
)
```

**Modes:**
- `GUARD` — defensive perimeter; sight cones scan outward for threats
- `FEED` — supply hub; GIVE cones converge on center piece

Uses Hungarian matching. Respects `effective_sight` — pieces won't be placed
farther than they can see.

### retreat

Scatter squad away from a threat. Each piece picks its own best retreat hex
independently (no formation maintained during withdrawal).

```python
orders = squad.retreat(
    threat_hex, grid, elevations,
    distance=5,               # max rings to search
    countries=None,
    food_tiers=None,
    sync=True,
    elevation_weight=0.5,     # preference for high ground
)
```

All pieces face **away** from the threat on arrival. Typical follow-up:

```python
# After executing retreat orders...
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### retreat_path (standalone)

Single-piece retreat. Returns a `TroopPath`.

```python
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)
```

Runs one Dijkstra flood-fill within `distance` rings. Scores candidates by:
```
score = elevation × elevation_weight − path_cost
```
Only considers hexes in the **away hemisphere** (cube dot product < 0 with
the toward-direction). Falls back to cheapest reachable hex if cornered.

---

## Synchronization

### sync_paths

Pad instruction lists so all pieces finish on the same turn.

```python
synced = sync_paths(
    pieces,                   # all pieces referenced by orders
    orders,                   # {piece_id: InstructionList}
    end_hexes=None,           # {piece_id: final_hex} for harvest check
    food_tiers=None,          # enables productive padding
    harvest_threshold=3,      # min food tier for harvest padding
    min_turns=0,              # floor (use leader's turns as floor)
)
```

**Padding priority:**
1. If `food_tiers[end_hex] >= threshold` → `HARVEST, GIVE, HARVEST, GIVE...`
2. Otherwise → `PAUSE, PAUSE, PAUSE...`

Called automatically by `form_up`, `march_to`, `surround`, and `retreat`
when `sync=True`.

---

## Helper Functions

```python
# Direction from one hex toward another (returns 0–5)
facing = _facing_toward(grid, from_idx, to_idx)

# Opposite direction (180°)
facing = _facing_away(grid, from_idx, to_idx)

# Effective sight with elevation bonus
sight = piece.effective_sight(elevations, elevation_mult=0.005)

# Formation slot offsets in cube coordinates
offsets = _formation_offsets(FormationType.WEDGE, n_followers, leader_facing)
```

---

## Recommended Combos

### Escort a Queen to a Settlement Site

```python
# Queen paths to destination
queen.instructions = queen.pathfind_to_rules(site, grid, elevations)

# Escort forms up around queen, then marches together
orders = escort_squad.march_to(
    site, FormationType.BOX, grid, elevations,
    leader=queen, food_tiers=food_tiers,
)
for p in escort_squad.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
```

### Defend a Settlement

```python
orders = garrison.surround(
    settlement.location, grid, elevations,
    mode=SurroundMode.GUARD, max_ring=2,
)
```

### Supply Chain: Harvest + Deliver

```python
# Pawn finds food, paths there, harvests, returns to queen
food_path = pawn.pathfind_to_food(grid, elevations, food_tiers)
return_path = pawn.pathfind(queen.location, grid, elevations,
                            source=food_path.end)

rules = (food_path.to_rules(grid, pawn.facing)
         + [Instruction.HARVEST.value] * 3
         + return_path.to_rules(grid, _facing_toward(grid, food_path.end, queen.location))
         + [Instruction.GIVE.value])

pawn.instructions = InstructionList(rules, patrol=True)
```

### Retreat + Regroup

```python
# Phase 1: scatter away from threat
retreat_orders = squad.retreat(enemy_hex, grid, elevations, distance=6)
for p in squad.alive:
    if p.id in retreat_orders:
        p.instructions = retreat_orders[p.id]

# Phase 2 (after retreat executes): regroup in defensive line
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### Pincer: Two Squads from Opposite Sides

```python
# Left wing approaches from the west
left_orders = left_squad.march_to(
    west_of_target, FormationType.LINE, grid, elevations)

# Right wing approaches from the east
right_orders = right_squad.march_to(
    east_of_target, FormationType.LINE, grid, elevations)

# Sync both wings to arrive simultaneously
all_pieces = left_squad.alive + right_squad.alive
all_orders = {**left_orders, **right_orders}
synced = sync_paths(all_pieces, all_orders)
```

### Scout Patrol (Loop)

```python
# Knight patrols a triangle: forward, turn, forward, turn...
knight.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
], patrol=True)  # loops forever
```

---

## Design Notes

**Hungarian matching** (`scipy.optimize.linear_sum_assignment`) is used
everywhere pieces need optimal assignment to positions: surround, form_up,
march_to. Cost matrix is `pathfind_cost[piece, slot]`. Complexity O(n³)
but n is squad size (typically 3–8), so it's instant.

**Occupied set is deliberately omitted** from plan overlays. Two pieces
can path through the same hex in preview. This is intentional — plans are
pre-execution previews, and collision resolution happens during `simulate()`.

**`form_up` vs `march_to`:**
- `form_up` — positions followers around leader's CURRENT hex. Leader not included in orders.
- `march_to` — positions everyone around a DESTINATION hex. Leader IS included in orders.

**`sync_paths` is composable.** Any function that returns
`dict[str, InstructionList]` can be synced. You can also sync orders from
different sources by merging the dicts before calling `sync_paths`.

**`retreat` breaks formation intentionally.** Each piece finds its own
best escape hex. Regroup afterward with `form_up`. Trying to maintain
formation during retreat is expensive and usually tactically wrong.

**Elevation matters in two places:**
1. Movement cost: uphill = 2×
2. Sight range: `effective_sight = base_sight + elevation × 0.005`

**Food integration:** All sync functions accept `food_tiers`. When a
piece's end hex has good food, slack turns become HARVEST+GIVE cycles
instead of idle PAUSEs. This means early-arriving units productively
farm while waiting for slower friends.
```

This covers the full API surface, the key design decisions, and the most useful composition patterns. Drop it in your `docs/` folder or as a `MANEUVERS.md` at the project root.

In [ ]:
#| export

@patch
def surround(self: Squad, target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign squad members to surround positions using Hungarian matching.
    
    Args:
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    pieces = self.alive
    if not pieces:
        return {}

    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))

    if not candidates:
        return {}

    n_pieces = len(pieces)
    n_positions = len(candidates)

    # 2. Build cost matrix, caching TroopPaths to avoid double-pathfind
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    paths: dict[tuple[int, int], TroopPath] = {}

    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        eff_sight = piece.effective_sight(elevations, elevation_mult)

        for j, (pos_idx, ring_dist) in enumerate(candidates):
            if ring_dist > eff_sight:
                continue
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue
            paths[(i, j)] = path
            cost[i, j] = path.cost

    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)

    # 4. Generate instruction lists from cached paths
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue

        piece = pieces[i]
        path = paths[(i, j)]
        pos_idx = candidates[j][0]

        rules = path.to_rules(grid, piece.facing)

        # Desired final facing
        if mode == SurroundMode.FEED:
            desired = _facing_toward(grid, pos_idx, target)
        else:
            desired = _facing_away(grid, pos_idx, target)

        # Facing after the last move step
        end_facing = _facing_toward(grid, path[-2], path[-1])

        # Shortest rotation to desired facing
        diff = (desired - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)

    return assignments


# Tactical Systems for Hex Strategy

Comprehensive tactical AI and algorithms for hex-based warfare and logistics.

## Overview

This document catalogs tactical patterns, formation algorithms, and strategic concepts for coordinating multiple pieces on a hex grid. Each tactic is backed by algorithms from graph theory, optimization, and computational geometry.

---

## Implemented Tactics

### 1. Surround (FEED/GUARD Modes)

**Purpose:** Position pieces around a high-value target (queen, settlement, chokepoint).

**Algorithm:** Hungarian matching (linear sum assignment)

```python
surround(pieces, target, grid, elevations,
         mode=SurroundMode.GUARD,  # or FEED
         max_ring=3)
```

**Implementation:**
1. Generate candidate positions (rings 1..max_ring around target)
2. Build cost matrix: `cost[piece, position] = pathfind_cost(piece → position)`
3. Constraint: `effective_sight(piece) >= distance(position, target)`
4. `linear_sum_assignment(cost)` → optimal piece↔position pairing
5. Generate paths + final facing (inward for FEED, outward for GUARD)

**Modes:**
- **FEED** — Face inward, supply chain formation, GIVE cones converge on center
- **GUARD** — Face outward, defensive perimeter, sight cones scan threats

**Complexity:** O(n³) for Hungarian, O(nm log m) for pathfinding (n=pieces, m=hexes)

**Strategic Uses:**
- Protect high-value piece (GUARD queen while she harvests)
- Establish supply hub (FEED formation feeds queen who distributes)
- Hold chokepoint (GUARD formation watches all approaches)

---

## Proposed Tactics

### 2. Pincer / Flanking

**Purpose:** Attack from multiple directions to divide enemy attention and cut retreat.

**Algorithm:** Two-group surround with asymmetric targeting

**Concept:**
- Split attackers into left and right wings
- Each wing surrounds one flank of target's position
- Wings converge from opposite sides (±90° from direct approach)

**Implementation:**
```python
def pincer(pieces, target, grid, elevations,
           flank_angle=90,    # degrees from center line
           attack_rings=2):   # how close to get
    """
    Flank from both sides.
    
    1. Split pieces into left/right wings (balanced by strength)
    2. Generate left-flank positions (target + offset in left direction)
    3. Generate right-flank positions (target + offset in right direction)
    4. Hungarian match each wing to its positions
    5. All pieces face toward target for coordinated assault
    """
```

**Algorithm Details:**
1. Compute center line from center-of-mass of pieces to target
2. Rotate ±`flank_angle` degrees to get left/right approach vectors
3. Generate positions along arcs at `attack_rings` distance from target
4. Partition pieces: high-attack units to wings, support units to center
5. Two independent Hungarian problems (left wing, right wing)
6. Synchronize movement so wings arrive simultaneously

**Strategic Uses:**
- Overwhelm defensive formation
- Cut off retreat path
- Force enemy to split attention between two fronts

---

### 3. High Ground Control

**Purpose:** Claim elevated positions for sight/defense advantage.

**Algorithm:** Elevation-weighted greedy assignment

**Concept:**
- Elevation grants sight bonus (`sight += elevation × 0.005`)
- Higher hexes see farther → better for archers, scouts, GIVE range
- Defensive advantage (enemy climbs uphill → slower, visible)

**Implementation:**
```python
def seize_high_ground(pieces, region: HexRegion, grid, elevations):
    """
    Assign pieces to high-elevation hexes in region.
    
    1. Rank all hexes in region by elevation
    2. Take top N hexes (N = number of pieces)
    3. Build cost matrix: pathfind cost + elevation penalty
       cost[i,j] = path_cost - elevation[j] * ELEV_WEIGHT
    4. Hungarian match pieces to high ground
    5. Generate paths, face outward for visibility
    """
```

**Scoring Function:**
```python
score(hex) = elevation * ELEV_WEIGHT - path_cost
```

**Strategic Uses:**
- Scouting — see enemy movements from peaks
- Artillery position — long-range pieces get max effective sight
- Defensive strongpoint — force enemy to attack uphill

---

### 4. Supply Line / Relay Chain

**Purpose:** Establish GIVE chain from harvesters to frontline.

**Algorithm:** Steiner tree approximation + flow optimization

**Concept:**
- Harvesters at food sources GIVE to relays
- Relays GIVE to frontline consumers
- Minimize total path length while ensuring connectivity

**Implementation:**
```python
def supply_chain(harvesters, consumers, grid, elevations, food_tiers):
    """
    Build relay chain connecting harvesters → consumers.
    
    1. Find high-yield food hexes (food_tier >= threshold)
    2. Assign harvesters to food hexes (nearest-match)
    3. Compute Steiner tree connecting harvesters + consumers
    4. Place relays at Steiner vertices
    5. Orient each piece's GIVE cone toward next hop in chain
    """
```

**Steiner Tree Approximation:**
1. Build minimum spanning tree (MST) on (harvesters ∪ consumers)
2. Use Dijkstra to compute shortest paths between all pairs
3. MST edges become supply routes
4. Relays positioned at branch points

**Flow Constraints:**
- Each relay must have `food_capacity >= downstream_demand`
- GIVE cones must cover next hop (facing check)
- Reserve `4 × diet` for self

**Strategic Uses:**
- Sustain prolonged siege
- Feed distant expansion force
- Enable low-harvest pieces (knights) to stay at front

---

### 5. Patrol Routes (Gosper Curve)

**Purpose:** Efficiently cover area without backtracking.

**Algorithm:** Space-filling curve generation

**Concept:**
- Gosper curve is hex-native space-filling curve
- Covers region with minimal repeated hexes
- Natural for patrol, scouting, area denial

**Implementation:**
```python
def patrol_route(anchor: int, radius: int, grid: HexGrid,
                 curve_order=2) -> list[int]:
    """
    Generate Gosper curve patrol route.
    
    1. Build Gosper curve of order N around anchor
    2. Filter to hexes within radius
    3. Convert curve to hex indices
    4. Return as InstructionList (path_to_rules)
    """
```

**Patrol Behaviors:**
- **SCOUT** — high sight pieces on wide patrol (order 3-4)
- **PERIMETER** — patrol kingdom border (anchor = capital)
- **SWEEP** — search for enemy in region

**Strategic Uses:**
- Early game exploration
- Border security
- Resource scouting

---

### 6. Follow the Leader

**Purpose:** Move formation relative to leader's position.

**Algorithm:** Offset maintenance

**Concept:**
- Designate one piece as leader
- Other pieces maintain fixed offset from leader
- Formation moves as a rigid body

**Implementation:**
```python
def follow_formation(leader: Piece, followers: list[Piece],
                     grid: HexGrid, offsets: dict[str, HexPosition]):
    """
    Maintain formation relative to leader.
    
    Args:
        offsets: piece.id → desired HexPosition offset from leader
    
    Each turn:
    1. Compute target = leader.location + offset
    2. If piece not at target: pathfind_to_rules(target)
    3. Face same direction as leader
    """
```

**Formation Templates:**
- **Line** — pieces in row behind leader
- **Wedge** — V-shape with leader at point
- **Box** — surround leader at ring-1
- **Column** — single file (for narrow passages)

**Strategic Uses:**
- Advance through difficult terrain as unit
- Retreat without breaking formation
- Convoy protection (civilians surrounded by guards)

---

### 7. Concentration of Force

**Purpose:** Mass pieces at single point for overwhelming attack.

**Algorithm:** Convergent pathfinding with timing

**Concept:**
- All pieces path to same target hex
- Arrivals synchronized (slower pieces start earlier)
- Coordinated assault on specific turn

**Implementation:**
```python
def converge(pieces, target, grid, elevations, assault_turn):
    """
    Coordinate multi-turn convergence.
    
    1. For each piece: pathfind to target
    2. Compute arrival_turn = current_turn + path_length / move_speed
    3. If arrival_turn < assault_turn: add PAUSE instructions
    4. If arrival_turn > assault_turn: piece too slow, assign alternate target
    5. All pieces execute simultaneously
    """
```

**Timing Constraint:**
```python
start_turn[piece] = assault_turn - path_length / move_strength
```

**Strategic Uses:**
- Coordinated siege
- Overwhelm isolated enemy
- Capture key objective (e.g., settlement)

---

### 8. Leapfrog Advance

**Purpose:** Alternating waves advance while covering each other.

**Algorithm:** Staged movement with overwatch

**Concept:**
- Group A advances while Group B provides sight/defense
- Group B advances while Group A sets up
- Repeat, trading roles each turn

**Implementation:**
```python
def leapfrog(vanguard: list[Piece], rearguard: list[Piece],
             destination: int, grid, elevations):
    """
    Alternating advance.
    
    Turn 1: vanguard moves forward, rearguard stays (DEFEND/GIVE)
    Turn 2: rearguard moves past vanguard, vanguard stays
    Repeat until both groups reach destination
    """
```

**Variant: Bounding Overwatch**
- Rearguard always has line-of-sight to vanguard
- GIVE cones cover vanguard's advance
- Rearguard includes high-sight pieces

**Strategic Uses:**
- Advance through contested territory
- Maintain supply chain during movement
- Minimize exposure to ambush

---

### 9. Defensive Line / Phalanx

**Purpose:** Hold position against frontal assault.

**Algorithm:** Line formation with mutual support

**Concept:**
- Pieces form contiguous line
- Each piece covers neighbors with GIVE/sight
- High-defense pieces at front, support at back

**Implementation:**
```python
def phalanx(pieces, line_start, line_end, grid, elevations):
    """
    Form defensive line.
    
    1. Generate hex positions along line from start to end
    2. Sort pieces: high HP front, high harvest_strength back
    3. Hungarian match pieces to line positions
    4. Face perpendicular to line (outward)
    5. GIVE cones point along line for mutual support
    """
```

**Line Generation:**
- Bresenham-like hex line algorithm
- Thicken to 2-3 hexes deep if enough pieces

**Variants:**
- **Shield Wall** — all pieces face same direction (enemy approach)
- **Hedgehog** — pieces face alternating directions (all-around defense)

**Strategic Uses:**
- Hold chokepoint (valley, bridge)
- Defend settlement perimeter
- Block enemy advance

---

### 10. Feint / Bait

**Purpose:** Lure enemy out of position with fake attack.

**Algorithm:** Pathfinding with retreat trigger

**Concept:**
- Small force advances toward enemy
- When enemy responds, retreat to prepared position
- Main force attacks exposed flank

**Implementation:**
```python
def feint(bait_pieces, main_force, enemy_position,
          grid, elevations, retreat_distance=5):
    """
    Lure enemy with sacrifice pieces.
    
    Bait group:
    1. Advance toward enemy (2/3 distance)
    2. If enemy within sight: retreat to rally point
    3. Rally point = starting position + retreat_distance
    
    Main force:
    1. Position at flanking angle from rally point
    2. Wait for bait to pull enemy
    3. Attack enemy flank when exposed
    """
```

**Trigger Condition:**
```python
if any(enemy in bait.pieces_in_sight(...)):
    execute_retreat()
```

**Strategic Uses:**
- Draw enemy from fortified position
- Split enemy formation
- Exhaust enemy movement (chasing bait)

---

## Advanced Tactical Concepts

### 11. Envelopment

**Purpose:** Surround enemy from all sides, cut off retreat.

**Algorithm:** Multi-group surround with rear blocker

**Concept:**
- Combine flanking (sides) + rearguard block (escape route)
- Three groups: left, right, rear
- Front group pins enemy while sides/rear close

**Implementation:**
3 simultaneous Hungarian problems:
1. Left wing → left flank positions
2. Right wing → right flank positions  
3. Rear group → positions behind enemy

**Historical Example:** Cannae (Hannibal)

---

### 12. Refuse Flank

**Purpose:** Concentrate force on one flank while refusing other.

**Algorithm:** Asymmetric deployment

**Concept:**
- Strong wing advances (attack)
- Weak wing retreats slowly (economy of force)
- Enemy's advance on weak wing overextends

**Implementation:**
```python
strong_wing: attack_positions (close)
weak_wing:   defensive_positions (far back)
```

**Historical Example:** Leuthen (Frederick the Great)

---

### 13. Hammer and Anvil

**Purpose:** Fix enemy against obstacle while attacking from mobility.

**Algorithm:** Pin + crush

**Concept:**
- Anvil: immobile force holds enemy in place
- Hammer: mobile force attacks from flank/rear
- Enemy crushed between two forces

**Implementation:**
```python
anvil_pieces: phalanx(enemy_front)
hammer_pieces: pincer(enemy_rear)
```

---

### 14. Interior Lines

**Purpose:** Use central position to defeat enemies in detail.

**Algorithm:** Sequential force concentration

**Concept:**
- Friendly forces centrally positioned
- Enemies approaching from multiple directions
- Defeat each enemy force before others arrive

**Implementation:**
```python
for enemy_group in sorted_by_arrival_time:
    converge(all_pieces, enemy_group.location)
    defeat(enemy_group)
    reposition_to_center()
```

---

### 15. Withdrawal / Fighting Retreat

**Purpose:** Preserve force while breaking contact.

**Algorithm:** Phased retreat with rearguard

**Concept:**
- Main body retreats
- Rearguard delays pursuers
- Rearguard leapfrogs through main body when pressed

**Implementation:**
```python
while distance(main_body, enemy) < safe_distance:
    main_body.move_away()
    rearguard.defend()
    if rearguard_threatened:
        rearguard.leap_through_main_body()
```

---

## Algorithms Catalog

Beyond the tactics above, here are useful algorithms for strategy games:

### Graph Algorithms

#### 1. **Dijkstra's Algorithm** ✅ (Implemented)
- **Use:** Shortest path with weighted edges
- **Application:** Pathfinding with elevation costs
- **Complexity:** O((V+E) log V) with priority queue

#### 2. **A* Search**
- **Use:** Heuristic-guided shortest path
- **Application:** Faster pathfinding when destination known
- **Heuristic:** Euclidean distance (hex coords)
- **Complexity:** O(E log V) typical, O(V log V) worst

#### 3. **Bidirectional Search**
- **Use:** Meet-in-the-middle pathfinding
- **Application:** Long-distance paths, reduces search space
- **Complexity:** O(√V) instead of O(V)

#### 4. **Jump Point Search (JPS)**
- **Use:** Fast pathfinding on uniform grids
- **Application:** Large maps with few obstacles
- **Caveat:** Requires adaptation for hex grids
- **Complexity:** O(E log V) but with smaller constant

#### 5. **Bellman-Ford Algorithm**
- **Use:** Shortest paths with negative edges
- **Application:** Modeling bonuses (friendly territory = negative cost)
- **Complexity:** O(VE)

#### 6. **Floyd-Warshall Algorithm**
- **Use:** All-pairs shortest paths
- **Application:** Precompute distance matrix for small maps
- **Complexity:** O(V³)

#### 7. **Minimum Spanning Tree (MST)** - Kruskal/Prim
- **Use:** Connect nodes with minimum total edge weight
- **Application:** Supply line network, road building
- **Complexity:** O(E log V)

#### 8. **Steiner Tree Approximation**
- **Use:** Connect terminals through intermediate points
- **Application:** Multi-drop supply routes (harvesters → relays → consumers)
- **Complexity:** NP-hard, 2-approximation in O(V³)

#### 9. **Maximum Flow (Ford-Fulkerson, Dinic)**
- **Use:** Find maximum throughput in network
- **Application:** Supply capacity limits, troop movement bandwidth
- **Complexity:** O(VE²) Ford-Fulkerson, O(V²E) Dinic

#### 10. **Minimum Cut (Max-Flow Min-Cut)**
- **Use:** Find weakest break in network
- **Application:** Identify chokepoints, vulnerable supply routes
- **Complexity:** Same as max-flow

#### 11. **Strongly Connected Components (Tarjan's)**
- **Use:** Find mutually reachable regions
- **Application:** Identify isolated territory, cut-off regions
- **Complexity:** O(V+E)

#### 12. **Topological Sort**
- **Use:** Order nodes in DAG
- **Application:** Tech tree dependencies, instruction sequencing
- **Complexity:** O(V+E)

---

### Optimization Algorithms

#### 13. **Hungarian Algorithm** ✅ (Implemented)
- **Use:** Minimum-cost bipartite matching
- **Application:** Assign pieces to positions optimally
- **Complexity:** O(n³)

#### 14. **Hopcroft-Karp Algorithm**
- **Use:** Maximum cardinality bipartite matching
- **Application:** Maximum piece assignments (ignoring cost)
- **Complexity:** O(E√V)

#### 15. **Linear Programming (Simplex)**
- **Use:** Optimize linear objective with constraints
- **Application:** Resource allocation, production planning
- **Complexity:** Exponential worst-case, polynomial average

#### 16. **Integer Linear Programming (ILP)**
- **Use:** LP with integer constraints
- **Application:** Discrete decisions (assign this piece? yes/no)
- **Complexity:** NP-hard, branch-and-bound practical

#### 17. **Dynamic Programming**
- **Use:** Optimal substructure problems
- **Application:** Knapsack (carry weight), longest common subsequence
- **Complexity:** Pseudo-polynomial, often O(nW) for knapsack

#### 18. **Greedy Algorithms**
- **Use:** Locally optimal choices
- **Application:** Huffman coding, activity selection
- **Example:** High-ground capture (pick highest hex iteratively)

---

### Geometric Algorithms

#### 19. **Convex Hull (Graham Scan)**
- **Use:** Smallest convex shape enclosing points
- **Application:** Identify territory boundary, detect encirclement
- **Complexity:** O(n log n)

#### 20. **Voronoi Diagram**
- **Use:** Partition plane by nearest-point regions
- **Application:** Territory influence, nearest-settlement zones
- **Complexity:** O(n log n)

#### 21. **Delaunay Triangulation**
- **Use:** Dual of Voronoi, maximizes minimum angle
- **Application:** Road network (edges), natural borders
- **Complexity:** O(n log n)

#### 22. **Line-of-Sight (Bresenham/DDA)**
- **Use:** Check visibility between two points
- **Application:** Can piece see target? (with terrain blocking)
- **Complexity:** O(distance)

#### 23. **Field-of-View (Shadow Casting)** ✅ (Implemented)
- **Use:** All visible hexes from viewpoint
- **Application:** `field_of_view()` for cone generation
- **Complexity:** O(sight²)

#### 24. **Coverage Sets** ✅ (Mentioned)
- **Use:** Minimum pieces to cover all hexes
- **Application:** Watchtower placement, minimum garrison
- **Complexity:** NP-hard (set cover), greedy approximation

---

### Computational Geometry

#### 25. **Point-in-Polygon**
- **Use:** Is hex inside region?
- **Application:** Territory checks, region queries
- **Complexity:** O(n) ray-casting

#### 26. **Polygon Clipping (Sutherland-Hodgman)**
- **Use:** Intersect two polygons
- **Application:** Overlapping territory, region intersection
- **Complexity:** O(n+m)

#### 27. **Sweep Line Algorithm**
- **Use:** Process events in spatial order
- **Application:** Intersection detection, range queries
- **Complexity:** O(n log n)

---

### Simulation & AI

#### 28. **Monte Carlo Tree Search (MCTS)**
- **Use:** Best move via random simulation
- **Application:** Strategy game AI, AlphaGo-style planning
- **Complexity:** Time-bounded

#### 29. **Minimax with Alpha-Beta Pruning**
- **Use:** Adversarial game tree search
- **Application:** Chess-like tactical decisions
- **Complexity:** O(b^d) branching^depth, pruning reduces

#### 30. **Influence Maps**
- **Use:** Heatmap of control/danger
- **Application:** Identify safe zones, contested areas
- **Complexity:** O(V) diffusion

#### 31. **Behavior Trees**
- **Use:** Hierarchical AI decision structure
- **Application:** Complex piece behaviors (if starving → seek food, else patrol)
- **Complexity:** O(tree depth)

---

### Heuristics & Approximations

#### 32. **Hill Climbing**
- **Use:** Local search optimization
- **Application:** Refine formation positions
- **Complexity:** O(iterations × neighbors)

#### 33. **Simulated Annealing**
- **Use:** Probabilistic local search (escapes local optima)
- **Application:** Large-scale unit positioning
- **Complexity:** O(iterations)

#### 34. **Genetic Algorithms**
- **Use:** Evolutionary optimization
- **Application:** Learn piece behavior rules
- **Complexity:** O(generations × population)

---

## Tactical Primitives

These are building blocks used in larger tactics:

### Positioning
- **Adjacent** — next to target (ring 1)
- **Surround** — all 6 neighbors occupied
- **Flank** — 90° from enemy facing
- **Rear** — 180° from enemy facing

### Movement
- **Advance** — toward enemy/objective
- **Retreat** — away from threat
- **Reposition** — lateral shift
- **Rally** — converge on point

### Facing
- **Face toward** — orient at target
- **Face away** — orient opposite
- **Face formation** — all same direction
- **Face perimeter** — outward from center

### Actions
- **Hold** — DEFEND, no movement
- **Support** — GIVE to nearby
- **Harvest** — gather resources
- **Scout** — maximize sight coverage

---

## Integration with Game Systems

### Food System Integration

All tactics must consider food logistics:

**Harvest Stops:**
```python
if piece.food < piece.diet * 4:  # 4-turn reserve
    # Interrupt tactical move to harvest
    detour_to_food()
```

**Supply Lines:**
```python
tactical_formation() + supply_chain()
# Ensure frontline pieces get food from rear
```

**Starvation Risk:**
- Long marches require harvesters or relays
- Sieges need established supply
- Retreats may abandon food sources

### Elevation System Integration

Tactics must account for terrain:

**High Ground Bonus:**
```python
effective_sight = piece.sight + elevation * 0.005
```

**Uphill Movement Penalty:**
```python
move_cost = base_cost * (1 + elevation_diff * 0.1)
```

**Tactical Implications:**
- Archers on peaks
- Cavalry in valleys
- Defend hilltops

---

## Implementation Priorities

### Phase 1: Core (Implemented) ✅
- [x] Surround (FEED/GUARD)
- [x] Pathfinding (Dijkstra)
- [x] Pieces in sight (FOV + elevation)
- [x] Food mechanics

### Phase 2: Essential
- [ ] High ground control
- [ ] Supply chain
- [ ] Pincer/flanking
- [ ] Patrol routes (Gosper)

### Phase 3: Advanced
- [ ] Leapfrog advance
- [ ] Phalanx/defensive line
- [ ] Concentration of force
- [ ] Follow-the-leader

### Phase 4: Expert
- [ ] Feint/bait
- [ ] Envelopment
- [ ] Interior lines
- [ ] Fighting retreat

---

## Testing & Visualization

### Tactical Overlays

For debugging and demonstration:

```python
class TacticalOverlay:
    """
    Visualize tactical assignments on map.
    
    - Color pieces by assignment (blue=wing1, red=wing2)
    - Draw arrows showing movement paths
    - Show facing cones
    - Highlight objectives (target hex)
    """
```

### Metrics

Evaluate tactic effectiveness:

- **Coverage** — % of region under friendly sight
- **Concentration** — avg distance between pieces
- **Exposure** — % of pieces visible to enemy
- **Supply** — % of pieces within GIVE range of food
- **Response Time** — turns to reach objective

---

## Future Directions

### Machine Learning

Train piece behaviors:
- Reinforcement learning for tactic selection
- Neural network for position evaluation
- Genetic algorithms for rule evolution

### Dynamic Tactics

Adapt to battlefield conditions:
- Switch from GUARD to FEED when starving
- Break phalanx to pursue fleeing enemy
- Abandon high ground if flanked

### Combined Arms

Coordinate piece types:
- Knights pin, archers strike
- Pawns harvest, bishops distribute
- Rooks block, queen commands

---

## References

### Classical Military Theory
- **Sun Tzu** — The Art of War (interior lines, deception)
- **Carl von Clausewitz** — On War (concentration of force)
- **Liddell Hart** — Strategy (indirect approach)

### Game AI
- **Steering Behaviors** — Craig Reynolds (flocking, separation)
- **GOAP** — Goal-Oriented Action Planning
- **HTN Planning** — Hierarchical Task Networks

### Algorithms
- **Introduction to Algorithms** — CLRS (graph algorithms)
- **Computational Geometry** — de Berg et al.
- **Combinatorial Optimization** — Papadimitriou & Steiglitz

---

## Summary Table

| Tactic | Algorithm | Complexity | Strategic Goal |
|--------|-----------|------------|----------------|
| Surround | Hungarian | O(n³) | Protect/contain |
| Pincer | Split Hungarian | O(n³) | Flank attack |
| High Ground | Greedy + Hungarian | O(n² log n) | Sight advantage |
| Supply Chain | Steiner Tree | O(V³) | Sustain force |
| Patrol | Gosper Curve | O(n) | Area coverage |
| Follow Leader | Offset maintain | O(n) | Formation move |
| Converge | Dijkstra + timing | O(nm log m) | Mass attack |
| Leapfrog | Staged movement | O(n) | Safe advance |
| Phalanx | Line assignment | O(n²) | Hold position |
| Feint | Path + trigger | O(n log n) | Lure enemy |

---

## Next Steps

1. **Implement Phase 2 tactics** (high ground, supply chain, pincer)
2. **Build tactical overlay system** for visualization
3. **Create tactic composer** — combine primitives into complex strategies
4. **Add A* pathfinding** for performance
5. **Test on real scenarios** (siege, retreat, resource contest)

The tactical system is designed to be modular — each tactic is independent but composable. As the game evolves, new tactics can be added by mixing these primitives and algorithms.


### Formations

In [ ]:
#| export
def sync_paths(pieces: list[Piece],
               orders: dict[str, InstructionList],
               end_hexes: dict[str, int] = None,
               food_tiers: np.ndarray = None,
               harvest_threshold: int = 3,
               min_turns: int = 0,
               ) -> dict[str, InstructionList]:
    """Pad InstructionLists so all pieces finish on the same turn."""
    if not orders:
        return {}

    piece_map = {p.id: p for p in pieces}
    end_hexes = end_hexes or {}

    turns = {
        pid: il.est_turns(piece_map[pid].move_strength
                          if pid in piece_map else 1.0)
        for pid, il in orders.items()
    }
    max_turns = max([min_turns] + list(turns.values()))
    synced: dict[str, InstructionList] = {}

    for pid, il in orders.items():
        slack = max_turns - turns[pid]
        if slack <= 0:
            synced[pid] = il
            continue

        p      = piece_map.get(pid)
        speed  = p.move_strength if p else 1.0
        budget = int(slack * speed)

        end_idx = end_hexes.get(pid)
        if end_idx is None and p is not None:
            end_idx = p.location

        use_harvest = (
            food_tiers is not None
            and end_idx is not None
            and 0 <= end_idx < len(food_tiers)
            and int(food_tiers[end_idx]) >= harvest_threshold
        )

        if use_harvest:
            cycle = [Instruction.HARVEST.value, Instruction.GIVE.value]
            pad   = [cycle[i % 2] for i in range(budget)]
        else:
            pad = [Instruction.PAUSE.value] * budget

        synced[pid] = InstructionList(
            list(il.rules) + pad, cursor=il.cursor, patrol=il.patrol,
        )

    return synced


In [ ]:
#| export
class FormationType(Enum):
    LINE   = "line"    # row abreast one step behind leader
    WEDGE  = "wedge"   # V opening backward, leader at point
    BOX    = "box"     # ring-1 around leader (spills to ring-2 if > 6)
    COLUMN = "column"  # single file directly behind


def _formation_offsets(formation: FormationType,
                       n: int,
                       facing: int) -> list[HexPosition]:
    """Cube-coordinate offsets (relative to leader) for n followers."""
    dirs  = HexPosition.directions()
    back  = dirs[(facing + 3) % 6]
    right = dirs[(facing + 1) % 6]
    left  = dirs[(facing + 5) % 6]

    def mul(d: HexPosition, k: int) -> HexPosition:
        return HexPosition(d.q * k, d.r * k, d.s * k)

    def add(a: HexPosition, b: HexPosition) -> HexPosition:
        return HexPosition(a.q + b.q, a.r + b.r, a.s + b.s)

    offsets: list[HexPosition] = []

    if formation == FormationType.COLUMN:
        # Single file: 1 back, 2 back, 3 back …
        for i in range(1, n + 1):
            offsets.append(mul(back, i))

    elif formation == FormationType.LINE:
        # One step back; spread centre → ±1 → ±2 …
        for i in range(n):
            if   i == 0:       spread = 0
            elif i % 2 == 1:   spread =  (i + 1) // 2
            else:              spread = -(i       // 2)
            base = mul(back, 1)
            if   spread > 0:   offsets.append(add(base, mul(right,  spread)))
            elif spread < 0:   offsets.append(add(base, mul(left,  -spread)))
            else:              offsets.append(base)

    elif formation == FormationType.WEDGE:
        # Pairs fan back-and-to-the-sides: rank k → k*back ± k*side
        rank = 1
        while len(offsets) < n:
            offsets.append(add(mul(back, rank), mul(right, rank)))
            if len(offsets) < n:
                offsets.append(add(mul(back, rank), mul(left, rank)))
            rank += 1

    elif formation == FormationType.BOX:
        # Ring-1 in priority order: sides → rear diagonals → rear → front
        priority = [
            (facing + 1) % 6,   # right
            (facing + 5) % 6,   # left
            (facing + 2) % 6,   # right-rear
            (facing + 4) % 6,   # left-rear
            (facing + 3) % 6,   # rear
             facing,             # front (last resort)
        ]
        for i in range(min(n, 6)):
            offsets.append(dirs[priority[i]])
        for i in range(n - 6):  # overflow into ring-2
            offsets.append(mul(dirs[priority[i % 6]], 2))

    return offsets[:n]


In [ ]:
#| export
def sync_paths(pieces: list[Piece],
               orders: dict[str, InstructionList],
               end_hexes: dict[str, int] = None,
               food_tiers: np.ndarray = None,
               harvest_threshold: int = 3,
               min_turns: int = 0,
               ) -> dict[str, InstructionList]:
    """Pad InstructionLists so all pieces finish on the same turn."""
    if not orders:
        return {}

    piece_map = {p.id: p for p in pieces}
    end_hexes = end_hexes or {}

    turns = {
        pid: il.est_turns(piece_map[pid].move_strength
                          if pid in piece_map else 1.0)
        for pid, il in orders.items()
    }
    max_turns = max([min_turns] + list(turns.values()))
    synced: dict[str, InstructionList] = {}

    for pid, il in orders.items():
        slack = max_turns - turns[pid]
        if slack <= 0:
            synced[pid] = il
            continue

        p      = piece_map.get(pid)
        speed  = p.move_strength if p else 1.0
        budget = int(slack * speed)

        end_idx = end_hexes.get(pid)
        if end_idx is None and p is not None:
            end_idx = p.location

        use_harvest = (
            food_tiers is not None
            and end_idx is not None
            and 0 <= end_idx < len(food_tiers)
            and int(food_tiers[end_idx]) >= harvest_threshold
        )

        if use_harvest:
            cycle = [Instruction.HARVEST.value, Instruction.GIVE.value]
            pad   = [cycle[i % 2] for i in range(budget)]
        else:
            pad = [Instruction.PAUSE.value] * budget

        synced[pid] = InstructionList(
            list(il.rules) + pad, cursor=il.cursor, patrol=il.patrol,
        )

    return synced


In [ ]:
#| export
def retreat_path(piece: Piece, threat_idx: int,
                 grid: HexGrid, elevations: np.ndarray,
                 distance: int = 5,
                 countries: np.ndarray = None,
                 elevation_weight: float = 0.5) -> TroopPath:
    """Path away from threat_idx using a single Dijkstra flood-fill.

    Explores all hexes within `distance` rings, scores candidates in the
    away-facing hemisphere:
        score = elevation * elevation_weight - path_cost

    Falls back to the lowest-cost reachable hex if the hemisphere is empty
    (e.g. piece is cornered against the map edge).
    """
    if piece.location is None or piece.location < 0:
        return TroopPath([], 0.0)

    start         = piece.location
    toward_facing = _facing_toward(grid, start, threat_idx)
    toward_d      = HexPosition.directions()[toward_facing]

    # Single Dijkstra — explore up to distance rings
    pq      = [(0.0, start)]
    visited = set()
    parent  = {start: None}
    best    = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        rel      = grid.index_to_hexposition(current, start)
        hex_dist = max(abs(rel.q), abs(rel.r), abs(rel.s))
        if hex_dist >= distance:
            continue   # don't expand past the boundary

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb]   = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    # Score candidates — prefer away-hemisphere + high ground
    best_score   = -float('inf')
    best_idx     = -1
    fallback_idx = -1
    fallback_cost = float('inf')

    for idx in visited:
        if idx == start:
            continue

        rel = grid.index_to_hexposition(idx, start)
        dot = rel.q * toward_d.q + rel.r * toward_d.r + rel.s * toward_d.s

        # Track a fallback (farthest away, cheapest) in case hemisphere is empty
        if best[idx] < fallback_cost:
            fallback_cost = best[idx]
            fallback_idx  = idx

        if dot >= 0:
            continue   # toward the threat — skip for primary scoring

        elev  = max(0.0, float(elevations[idx]))
        score = elev * elevation_weight - best[idx]
        if score > best_score:
            best_score = score
            best_idx   = idx

    chosen = best_idx if best_idx >= 0 else fallback_idx
    if chosen < 0:
        return TroopPath([], 0.0)

    # Reconstruct path
    path, node = [], chosen
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return TroopPath(path, best[chosen])


In [ ]:
#| export
@patch
def march_to(self: Squad,
             destination: int,
             formation: FormationType,
             grid: HexGrid,
             elevations: np.ndarray,
             leader: Piece = None,
             countries: np.ndarray = None,
             food_tiers: np.ndarray = None,
             sync: bool = True,
             ) -> dict[str, InstructionList]:
    """Move the entire squad to a new anchor hex in formation.

    Unlike form_up (which slots around the leader's *current* position),
    march_to computes formation slots relative to the *destination*, using
    the leader's arrival facing — so the formation assembles correctly on
    arrival rather than during the march.

    Returns:
        {piece_id: InstructionList} for ALL pieces including the leader.
        Returns empty dict if the leader has no path to destination.
    """
    alive = self.alive
    if not alive:
        return {}

    if leader is None:
        leader = self.by_rank()[0]

    followers = [p for p in alive if p.id != leader.id]

    # 1. Leader's path
    leader_path = leader.pathfind(destination, grid, elevations, countries)
    if not leader_path:
        return {}

    leader_rules = leader_path.to_rules(grid, leader.facing)

    # Arrival facing derived from the last two hexes of the leader's route
    arrive_facing = (_facing_toward(grid, leader_path[-2], leader_path[-1])
                     if len(leader_path.hexes) >= 2 else leader.facing)

    # 2. Formation slots relative to destination + arrival facing
    offsets = _formation_offsets(formation, len(followers), arrive_facing)

    targets: list[int | None] = []
    for off in offsets:
        idx = grid.hexposition_to_index(off, destination)
        ok  = (0 <= idx < len(elevations)
               and idx not in grid.invalidRegion
               and elevations[idx] > 0
               and (countries is None or countries[idx] >= 0))
        targets.append(idx if ok else None)

    # 3. Cost matrix: followers × slots
    INF   = 1e9
    n     = len(followers)
    cost  = np.full((n, len(targets)), INF)
    paths: dict[tuple[int, int], TroopPath] = {}

    for i, piece in enumerate(followers):
        if piece.location is None or piece.location < 0:
            continue
        for j, target in enumerate(targets):
            if target is None:
                continue
            path = piece.pathfind(target, grid, elevations, countries)
            if path:
                paths[(i, j)] = path
                cost[i, j]    = path.cost

    row_ind, col_ind = linear_sum_assignment(cost)

    orders:    dict[str, InstructionList] = {
        leader.id: InstructionList(leader_rules, cursor=0, patrol=False),
    }
    end_hexes: dict[str, int] = {leader.id: destination}

    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        piece  = followers[i]
        path   = paths[(i, j)]
        target = targets[j]

        rules = path.to_rules(grid, piece.facing)

        # Rotate to match leader's arrival facing
        follower_arrive = (_facing_toward(grid, path[-2], path[-1])
                           if len(path.hexes) >= 2 else piece.facing)
        diff = (arrive_facing - follower_arrive) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        orders[piece.id]    = InstructionList(rules, cursor=0, patrol=False)
        end_hexes[piece.id] = target

    # 4. Sync everyone — leader included — to arrive the same turn
    if sync and orders:
        all_pieces = [leader] + followers
        orders = sync_paths(all_pieces, orders,
                            end_hexes=end_hexes,
                            food_tiers=food_tiers)
    return orders


In [ ]:
#| export
@patch
def retreat(self: Squad,
            threat_idx: int,
            grid: HexGrid,
            elevations: np.ndarray,
            distance: int = 5,
            countries: np.ndarray = None,
            food_tiers: np.ndarray = None,
            sync: bool = True,
            elevation_weight: float = 0.5,
            ) -> dict[str, InstructionList]:
    """Assign independent retreat paths to all living squad members.

    Each piece picks its own best retreat hex (no formation maintained
    during the withdrawal). Typical usage:

        orders = squad.retreat(threat, grid, elevations)
        # ... apply orders, execute turns ...
        regrouped = squad.form_up(FormationType.LINE, grid, elevations)
    """
    orders:    dict[str, InstructionList] = {}
    end_hexes: dict[str, int]             = {}

    for piece in self.alive:
        path = retreat_path(piece, threat_idx, grid, elevations,
                            distance=distance, countries=countries,
                            elevation_weight=elevation_weight)
        if not path:
            continue

        rules = path.to_rules(grid, piece.facing)

        # Face away from threat on arrival
        away_dir      = _facing_away(grid, path.end, threat_idx)
        arrive_facing = (_facing_toward(grid, path[-2], path[-1])
                         if len(path.hexes) >= 2 else piece.facing)
        diff = (away_dir - arrive_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        orders[piece.id]    = InstructionList(rules, cursor=0, patrol=False)
        end_hexes[piece.id] = path.end

    if sync and orders:
        orders = sync_paths(self.alive, orders,
                            end_hexes=end_hexes,
                            food_tiers=food_tiers)
    return orders


This brings up a great point. we are going to need a functions that synchronize a list of paths (padding with feeds and shares)

another that makes a formation ine — pieces in row behind leader
Wedge — V-shape with leader at point
Box — surround leader at ring-1
Column — single file (for narrow passages)


lets modify form up so that if no piece is given it picks the piece with the highest rank. I also moves est to be on InstructionList

Anything else you would add to formations and paths?

Lets build march_to and retreat_path

So I am going to need an api doc that will be helpful to other developers that summerizes how to use the Manuevers, what are some good combo's. The ai comments get erased when I do git commits so anything future developers (including me) need to remember please write

## Piece editor

In [ ]:
#| export
# ── 4. Route that receives the drag ───────────────────────────

@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return badge display."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # We need the grid to convert indices → HexPositions.
    # Use a fresh centered grid matching piece_drag_editor's rings/radius.
    rings  = 3
    radius = 35
    dummy_style = StyleCSS("x", fill="white")
    grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

    new_rules = path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist to DB if we can find the piece
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            merged = existing + new_rules   # append; swap for replace if preferred
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in merged)},
                pk="id"
            )
            new_rules = merged
    except Exception:
        pass   # demo mode — just show without saving

    dummy = Piece(id=piece_id, rules=new_rules)
    return rules_display(dummy.rules, piece_id)

In [ ]:
#| export
dummy_rook = Piece(
    id   = "demo-rook-0000-0000-000000000000",
    piece_type = PieceType.ROOK,
    facing     = 4,   # starts facing E
    name = "Lucia"
)

webMe(Div(
    H3("Rook Rule Editor"),
    piece_drag_editor(dummy_rook),
    cls="p-4 space-y-4 bg-slate-900 text-white"
))

Should the piece_drag_editor put a small icon in

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret

In [ ]:
showActiveSettlements(dummySession)

### Revisit

In [ ]:
#| export
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    """Live demo page for the drag-to-rules editor."""
    editor = piece_drag_editor(dummy_rook)
    facing = facing_grid(dummy_rook)
    return Titled("Piece Editor Demo",
        Div(
            H3("Rook Rule Editor"),
            P("Drag across hexes to create movement rules.",
              cls="text-sm opacity-70 mb-4"),
            editor,
            Divider(),
            H4("Facing Selector"),
            facing,
            cls="p-6 space-y-6 max-w-xl mx-auto"
        )
    )


@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, return compact badge strip."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # Reconstruct same grid as piece_drag_editor
    grid = HexGrid.centered(rings=3, radius=35,
                            style=StyleCSS("x", fill="white"))

    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    # For real pieces, merge with existing and persist
    if not piece_id.startswith("demo-"):
        try:
            rec = globalStore.pieces[piece_id]
            if rec:
                existing_str = rec.rules if hasattr(rec, 'rules') else ""
                existing = [int(x) for x in (existing_str or "").split(",") if x]
                new_rules = existing + new_rules
                globalStore.pieces.upsert(
                    {"id": piece_id, "rules": ",".join(str(r) for r in new_rules)},
                    pk="id",
                )
        except Exception as e:
            logging.warning(f"piece_path: DB skip: {e}")

    return InstructionList(new_rules).compact(piece_id)


In [ ]:
#| export
@rt("/piece_detail/{id}")
def piece_detail(session, id: str):
    """Full piece stats card for the right panel."""
    uid = _auth_piece(session, id)
    if uid is None:
        return _DENIED

    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", cls="text-sm opacity-50")

    piece = globalStore.piece_from_id(id)
    if piece is None:
        return P("Piece not found", cls="text-sm text-red-500")

    _wire_flag(piece, active.board)
    return piece  # delegates to Piece.__ft__


In [ ]:
#| export
@rt("/summit_demo")
def summit_demo(session):
    board = PieceBoard.hilly(rings=5, radius=22, seed=42)
    grid  = board.grid

    # Find a high hex near center for the summit
    mid = grid.middle
    elevs = board.elevations
    summit_idx = max(range(len(grid.hexes)),
                     key=lambda i: elevs[i] if abs(i - mid) < 15 else 0)

    # Kingdom 1 — NW corner, trying to take the hill
    k1 = [
        board.add_piece(mid - grid.nCols*3 - 2, PieceType.KNIGHT, facing=4,
                        country_id=1,
                        instructions=InstructionList([
                            Instruction.FORWARD.value]*4 + [Instruction.ROT_R.value])),
        board.add_piece(mid - grid.nCols*2 - 3, PieceType.PAWN, facing=3,
                        country_id=1,
                        instructions=InstructionList([
                            Instruction.FORWARD.value,
                            Instruction.FORWARD.value,
                            Instruction.HARVEST.value])),
    ]

    # Kingdom 2 — SE corner, defending the hill
    k2 = [
        board.add_piece(mid + grid.nCols*3 + 2, PieceType.ROOK, facing=1,
                        country_id=2,
                        instructions=InstructionList([
                            Instruction.FORWARD.value]*3 + [Instruction.DEFEND.value])),
        board.add_piece(mid + grid.nCols*2 + 3, PieceType.BISHOP, facing=2,
                        country_id=2,
                        instructions=InstructionList([
                            Instruction.FORWARD.value,
                            Instruction.FORWARD.value,
                            Instruction.ROT_L.value,
                            Instruction.SETTLE.value])),
    ]

    # Mark the summit
    grid.hexes[summit_idx].label = "⛰"
    grid.hexes[summit_idx].labelStyle = "summit-lbl"
    grid.builder.add_style(StyleCSS("summit-lbl", fill="#fff", font_size="14px"))

    board_svg = board.render(show_plan=True, num_turns=4)

    country_names = {1: "Iron Keep", 2: "Ash Basin"}

    return Titled("Race to the Summit",
        Div(
            # Left — roster
            Div(
                H4("⚔ Armies", cls="font-bold mb-2"),
                GroupedPieceList(
                    board.pieces,
                    country_names=country_names,
                    country_flags=board.flags,
                    hx_get_prefix="/piece_detail",
                    hx_target="#right-panel",
                ),
                Divider(),
                P("Click a piece to edit its orders",
                  cls="text-xs opacity-50"),
                cls="w-56 flex-shrink-0 p-3 bg-base-200 overflow-y-auto space-y-2",
            ),

            # Center — board
            Div(
                NotStr(board_svg),
                id="map",
                cls="flex-1 overflow-auto",
            ),

            # Right — piece detail / editor
            Div(
                P("← Select a piece", cls="text-sm opacity-40 p-4"),
                id="right-panel",
                cls="w-64 flex-shrink-0 p-3 bg-base-200 overflow-y-auto",
            ),

            cls="flex gap-0", style="height:600px;",
        )
    )


In [ ]:
!tail -10 base.text

In [ ]:
webMe(summit_demo)

Can we build a demo with a piece on a pieceboard and the piece detail on the left. when the piece''s instruction set is changed the overlay/pieceboard changes

In [ ]:
#| export
# ── helper ─────────────────────────────────────────────────────────────

def _plan_board_svg(piece, active, num_turns=5):
    """Simulate the piece's patrol and return a zoomed board SVG string."""
    terrain = active.board.terrain
    grid    = terrain.hexGrid
    steps, region = piece.plan_region(
        grid, terrain.elevations,
        num_turns=num_turns, padding=4,
        countries=terrain.fields.get("country"),
    )
    result = active.cover.zoom_region_fast(region, compute_weather=True)
    z = result.terrain
    z.hexGrid.adjustRadius(25)
    z.colorMap()
    z.hexGrid.update()
    z.hexGrid.builder.adjust(
        "plan",
        piece_plan_overlay(piece, z.hexGrid, steps=steps, c2f=result.c2f),
    )
    return z.hexGrid.builder.xml()


# ── demo page ───────────────────────────────────────────────────────────

@rt("/live_piece_demo")  
def live_piece_demo(session):
    uid    = ensure_user(session)
    active = globalStore.active_board(uid)
    
    if active is None:
        # Same setup that the notebook does for dummySession
        globalStore.setup_new_game(uid)   # ← whatever that function is called
        active = globalStore.active_board(uid)
    
    if active is None:
        return Titled("Demo", P("Still no active board — check setup."))

    piece = next(
        (c for k in active.board.kingdoms
             for s in k.settlements
             for c in s.citizens),
        None,
    )
    if piece is None:
        return Titled("Demo", P("No piece found — run the board setup first."))

    # Ensure a non-empty patrol so the initial board renders something
    if not piece.instructions:
        piece.instructions = InstructionList(
            [Instruction.FORWARD.value, Instruction.FORWARD.value,
             Instruction.ROT_R.value,  Instruction.FORWARD.value],
            cursor=0, patrol=True,
        )

    pid8 = piece.id[:8]

    left = Div(
        # piece stats
        H3(piece.piece_type.name.title(), cls="font-bold text-lg mb-1"),
        Div(
            Span(f"⚔ {piece.attack_strength}", cls="badge badge-sm"),
            Span(f"🏃 {piece.move_strength}",  cls="badge badge-sm"),
            Span(f"👁 {piece.sight}",           cls="badge badge-sm"),
            Span(f"hex {piece.location}",       cls="badge badge-outline badge-sm"),
            cls="flex flex-wrap gap-1 mb-3",
        ),
        Divider(),
        H4("Draw Patrol Route", cls="font-semibold text-sm mb-1"),
        P("Drag across hexes to set movement orders.",
          cls="text-xs opacity-60 mb-2"),
        # standard drag editor — JS patch below redirects its post
        piece_drag_editor(piece),
        # instruction badge strip (id="rules-{pid8}", updated live)
        piece.instructions.compact(piece.id),
        cls="w-72 flex-shrink-0 p-4 bg-base-200 overflow-y-auto space-y-1",
    )

    right = Div(
        NotStr(_plan_board_svg(piece, active)),
        id="board-panel",
        cls="flex-1 overflow-auto p-2",
    )

    # Monkeypatch htmx.ajax so /piece_path/ → /live_piece_path/
    # (scoped to this iframe, harmless elsewhere)
    patch = Script("""
(function () {
    var _orig = htmx.ajax.bind(htmx);
    htmx.ajax = function (verb, url, opts) {
        if (url && url.indexOf('/piece_path/') !== -1)
            url = url.replace('/piece_path/', '/live_piece_path/');
        return _orig(verb, url, opts);
    };
})();
""")

    return Titled("Live Piece Plan Editor",
        Div(left, right, patch, cls="flex h-[700px]"),
    )


# ── live-update endpoint ────────────────────────────────────────────────

@rt("/live_piece_path/{piece_id}/{facing:int}")
async def live_piece_path(request, session, piece_id: str, facing: int):
    """Replace instructions in-memory; return rules badge + OOB board redraw."""
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    pid8    = piece_id[:8]

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]
    grid_sm = HexGrid.centered(rings=3, radius=35,
                               style=StyleCSS("x", fill="white"))
    rules   = InstructionList.path_to_rules(hex_ids, grid_sm, start_facing=facing)

    uid    = ensure_user(session)
    active = globalStore.active_board(uid)

    piece = next(
        (c for k in active.board.kingdoms
             for s in k.settlements
             for c in s.citizens
             if c.id == piece_id),
        None,
    )
    if piece is None:
        return Div(P("Piece not found", cls="text-error text-sm"), id=f"rules-{pid8}")

    # Replace (not append) the patrol
    piece.instructions = InstructionList(rules, cursor=0, patrol=True)

    return (
        # primary: outerHTML of #rules-{pid8}
        piece.instructions.compact(piece_id),
        # OOB: innerHTML of #board-panel
        Div(NotStr(_plan_board_svg(piece, active)),
            id="board-panel", hx_swap_oob="innerHTML"),
    )


webMe("/live_piece_demo")



# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: `grid.update()` is called once, flushing all hex style mutations into the `hexes` layer.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay()` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay()` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width)` | 60 | SVG | River network from drainage basins |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | Overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'}
# If basins=None, logs: "Skipping rivers: missing {'basins'}"
```

---

## Creating Custom Overlays

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Custom overlay with hex style mutations

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        # ctx.basins is guaranteed present because of requires
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Create custom | Write a function returning `OverlaySpec` |
```

In [ ]:
??Squad

can you build squadPlanOverlay which would be in the TerrainDisplay format.
It would take in a list of squads and build something similar as piece_plan_overlay for them.

In [ ]:
#| export
def SquadPlanOverlay(squads, num_turns=5, **kw) -> OverlaySpec:
    """Patrol-plan overlay for all pieces across multiple squads.

    Renders ghost pieces, dashed arrows, rotation arcs, and harvest icons
    for every piece with instructions set. Each piece gets unique SVG ids
    via its piece.id, so multiple squads compose cleanly.

    Args:
        squads:    list of Squad objects (uses .alive for living pieces)
        num_turns: how many turns to simulate ahead

    Context used:
        ctx.grid / ctx.terrain  — render grid (may be zoomed)
        ctx.board               — coarse grid for simulation (optional, falls back to ctx.grid)
        ctx.c2f                 — coarse-to-fine index mapping for zoomed views (optional)
    """

    def render(ctx) -> str:
        grid = ctx.grid
        c2f  = getattr(ctx, 'c2f', None)

        # Simulate on the coarse grid when a board is available,
        # otherwise fall back to the render grid (non-zoomed case).
        board = getattr(ctx, 'board', None)
        if board is not None:
            sim_grid = board.terrain.hexGrid
            sim_elev = board.terrain.elevations
            sim_ctry = board.terrain.fields.get("country")
        else:
            sim_grid = grid
            sim_elev = ctx.terrain.elevations
            sim_ctry = ctx.terrain.fields.get("country")

        parts = []
        for squad in squads:
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            for piece in pieces:
                if not piece.instructions or not piece.instructions.rules:
                    continue
                steps, _ = piece.plan_region(
                    sim_grid, sim_elev,
                    num_turns=num_turns, padding=0,
                    countries=sim_ctry,
                )
                # Filter steps to only those mappable into the zoomed region
                if c2f is not None:
                    steps = [s for s in steps if s.hex_idx in c2f]
                parts.append(
                    piece_plan_overlay(piece, grid, steps=steps, c2f=c2f)
                )

        return '\n'.join(parts)

    return OverlaySpec("squad_plan", render, priority=70)


there were some mapping coordinate bugs I had to fix in another overlay. I think SquadPlanOverlay has these as well

```
def SquadVisionOverlay(squads, num_turns=5, facing_only=True,
                       opacity=0.30, color_attr="darkPrimary",
                       elevation_mult=0.005, **kw) -> OverlaySpec:
    """Vision cones + facing bars at each piece's simulated final position.

    Renders underneath SquadPlanOverlay (priority 65 < 70):
      - Dotted hex fill for every hex in each piece's FOV
      - White-on-dark facing bar showing which way each piece ends up looking

    Args:
        squads:         list of Squad objects
        num_turns:      turns to simulate forward
        facing_only:    True = 120° cone, False = full radius
        opacity:        fill opacity for vision hexes
        color_attr:     flag attribute for dot color ("darkPrimary", "primary", "comp")
        elevation_mult: sight bonus per elevation unit
    """

    def render(ctx) -> str:
        grid = ctx.grid
        c2f  = getattr(ctx, 'c2f', None)

        board = getattr(ctx, 'board', None)
        if board is not None:
            sim_grid = board.terrain.hexGrid
            sim_elev = board.terrain.elevations
            sim_ctry = board.terrain.fields.get("country")
        else:
            sim_grid = grid
            sim_elev = ctx.terrain.elevations
            sim_ctry = ctx.terrain.fields.get("country")

        # ── Simulate to collect final positions ──
        color_to_hexes: dict[str, set[int]] = {}
        bars = []  # (render_hex_idx, facing, flag)

        for squad in squads:
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            for piece in pieces:
                if not piece.instructions or not piece.instructions.rules:
                    continue

                steps, _ = piece.plan_region(
                    sim_grid, sim_elev,
                    num_turns=num_turns, padding=0,
                    countries=sim_ctry,
                )
                if not steps:
                    continue

                last = steps[-1]
                color = (getattr(piece.flag, color_attr, "#888")
                         if piece.flag else "#888")

                # Vision from final position
                elev = max(0, sim_elev[last.hex_idx])
                eff_sight = int(piece.sight + elev * elevation_mult)

                if facing_only:
                    facing_dir = HexPosition.directions()[last.facing % 6]
                    fov = HexPosition.origin().field_of_view( facing_dir, eff_sight)
                    visible = [idx for hp in fov
                               if (idx := sim_grid.hexposition_to_index(hp, last.hex_idx)) >= 0]
                else:
                    visible = sim_grid.indices_in_range(last.hex_idx, eff_sight)

                # Map coarse → fine when zoomed
                if c2f is not None:
                    visible = [c2f[v] for v in visible if v in c2f]
                    render_idx = c2f.get(last.hex_idx)
                else:
                    render_idx = last.hex_idx

                color_to_hexes.setdefault(color, set()).update(visible)

                if render_idx is not None and 0 <= render_idx < len(grid.hexes):
                    bars.append((render_idx, last.facing, piece.flag))

        # ── Render vision hex fill ──
        parts = []
        r = grid.radius if hasattr(grid, 'radius') else 20
        dot_r   = max(2.5, r * 0.18)
        spacing = max(5.0, r * 0.32)

        for color, hex_indices in color_to_hexes.items():
            pat_id = f"vision_{color.replace('#', '')}"
            pat_svg = (
                f'<pattern id="{pat_id}" x="0" y="0" '
                f'width="{spacing:.1f}" height="{spacing:.1f}" '
                f'patternUnits="userSpaceOnUse">'
                f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
                f'r="{dot_r:.1f}" fill="{color}"/>'
                f'</pattern>')
            ctx.builder.add_definition(SVGDef("", pat_id, pat_svg, raw=True))

            for idx in sorted(hex_indices):
                if idx < 0 or idx >= len(grid.hexes):
                    continue
                h = grid.hexes[idx]
                pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
                parts.append(
                    f'<polygon points="{pts}" '
                    f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                    f'stroke="{color}" stroke-width="0.6" '
                    f'stroke-dasharray="3,2" stroke-opacity="0.4"/>')

        # ── Render facing bars ──
        for render_idx, facing, flag in bars:
            c = grid.hexes[render_idx].center
            g_out = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
            g_bar = DiagramGlyphs(color="#FFFFFF", size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)
            parts.append(g_out.facing_bar(c.x, c.y, facing))
            parts.append(g_bar.facing_bar(c.x, c.y, facing))

        return '\n'.join(parts)

    return OverlaySpec("squad_vision", render, priority=65)
```

In [ ]:
from HexMagic.game.data import _map_point
??_map_point

can we do both?

so what should the full _map_point be?

can you write SquadPlanOverlay?

In [ ]:
#| export
def SquadSymbolOverlay(squads, size='board', font='Cinzel',
                       label_offset=22, show_label=True, **kw) -> OverlaySpec:
    """Draw each squad's animal symbol at its centroid with a name label beneath.

    Args:
        squads:       list of Squad objects
        size:         animal_svg size — 'board', 'list', or 'large'
        font:         font family for labels (loaded via add_font)
        label_offset: vertical px offset for label below symbol center
        show_label:   whether to render the text label

    Context used:
        ctx.grid / ctx.builder  — render grid
        ctx.board               — coarse grid for centroid calc (optional)
        ctx.c2f                 — coarse→fine mapping for zoomed views (optional)
    """

    def render(ctx) -> str:
        grid    = ctx.grid
        c2f     = getattr(ctx, 'c2f', None)
        board   = getattr(ctx, 'board', None)
        sim_grid = board.terrain.hexGrid if board else grid

        ctx.builder.add_font(font)

        parts = []
        for i, squad in enumerate(squads):
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            if not pieces:
                continue

            center_idx, _ = pieces_center(pieces, sim_grid)
            if center_idx < 0:
                continue

            # Map coarse → fine for zoomed views
            render_idx = _map_point(center_idx, c2f)
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            center = grid.hexes[render_idx].center
            flag   = squad.flag
            if flag is None:
                continue

            # ── Animal symbol ──
            sid = f"sq_{squad.id}_{i}"
            parts.append(flag.animal_svg(size=size, center=center, piece_id=sid))

            # ── Label ──
            if show_label:
                label = squad.name or squad.animal or ""
                if not label:
                    continue
                lbl_style = flag.labelStyle(f"sql_{squad.id}_{i}")
                ctx.builder.add_style(lbl_style)
                parts.append(
                    f'<text x="{center.x}" y="{center.y + label_offset}" '
                    f'text-anchor="middle" font-size="11" '
                    f"font-family=\"'{font}', serif\" "
                    f'dominant-baseline="hanging" '
                    f'class="{lbl_style.name}">{label}</text>'
                )

        return '\n'.join(parts)

    return OverlaySpec("squad_symbols", render, priority=75)


I suspect some of my map coordinate bugs are in SquadSymbolOverlay from trying to make it work in a zoomed context.
```
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[190], line 3
      1 # Zoomed
      2 ctx = OverlayContext.from_terrain(myTerr, region=region, padding=2)
----> 3 ctx.test(SquadSymbolOverlay([queenGuard]))

File ~/HexMagic/HexMagic/overlay.py:136, in OverlayContext.test(self, *overlays)
    134 if missing:
    135     raise ValueError(f"'{o.name}' requires {missing} but context has {self.available}")
--> 136 svg = o.renderer(self)
    137 tag = f"SVG ({len(svg)} chars)" if svg else "style-only"
    138 print(f"  ✓ {o.name:20s} pri={o.priority:<4d} {tag}")

File ~/HexMagic/HexMagic/game/piece.py:3255, in SquadSymbolOverlay.<locals>.render(ctx)
   3253 # Map coarse → fine for zoomed views
   3254 render_idx = c2f.get(center_idx, center_idx) if c2f else center_idx
-> 3255 if render_idx < 0 or render_idx >= len(grid.hexes):
   3256     continue
   3258 center = grid.hexes[render_idx].center

TypeError: '<' not supported between instances of 'list' and 'int'
```

can you write the proper SquadSymbolOverlay?

Is the vision overlay fixed?

Please write the proper SquadVisionOverlay

I would love SquadSymbolOverlay that would draw the squads symbol using pieces_center on the map. maybe the label underneath with the Cizel font or something similar. we definitely want the OverlaySpec and this was how we a names for the counties to see text
```
@patch
def names_overlay(self: GameBoard, terrain: Terrain = None,
                  c2f: dict = None) -> str:
    """Kingdom name labels."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    overlay = ""

    for country in self.kingdoms:
        if not country.countryName:
            continue

        region = country.region.project_region(grid, c2f)
        if not region.hexes:
            continue

        label_style = country.flag.labelStyle(f"n_{country.countryId}")
        #label_style.properties["fill"] = "#36454F"
        #label_style.properties["stroke"] = "white"
        #label_style.properties["stroke-width"] = "2"
        #label_style.properties["paint-order"] = "stroke"
        grid.builder.add_style(label_style)
        grid.builder.add_font('Cinzel')

        centroid_idx = region.centroid_hex()
        if centroid_idx >= 0:
            cx = grid.hexes[centroid_idx].center.x
            cy = grid.hexes[centroid_idx].center.y
            overlay += (
                f'\t<text x="{cx}" y="{cy}" text-anchor="middle" font-size="14" '
                f'font-family="\'Cinzel\', sans-serif" '
                f'dominant-baseline="middle" class="{label_style.name}">'
                f'{country.countryName}</text>\n'
            )

    return overlay
    ```

    and this was labels
    ```
    flags = CountryFlag.seaborn("husl", 5)
canvas = SVGBuilder()
canvas.add_font('Cinzel')

sizes = ['board', 'list', 'large']
scales = {'board': 0.5, 'list': 1.5, 'large': 3.5}
col_widths = [80, 120, 220]  # px per column
row_height = 230
margin = 20

canvas.width = sum(col_widths) + margin * 4
canvas.height = len(flags) * row_height + margin

for row, flag in enumerate(flags):
    animal = flag.animal_name()
    cx_offset = margin
    cy = margin + row * row_height + 110
    
    for col, size in enumerate(sizes):
        scale = scales[size]
        r = 22.5 * scale * 1.3
        cx = cx_offset + col_widths[col] // 2
        
        svg = flag.animal_svg(size=size, center=MapCord(cx, cy),
                              piece_id=f"a_{row}_{col}")
        canvas.adjust(f"animal_{row}_{col}", svg)
        cx_offset += col_widths[col] + margin
    
    # Label
    lbl = flag.labelStyle(f"lbl_{row}")
    canvas.add_style(lbl)
    canvas.adjust(f"name_{row}",
        f'<text x="{canvas.width - 10}" y="{cy}" text-anchor="end" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{flag.name} — {animal}</text>')

canvas.show()
```